2007


In [ ]:
# ==============================================================================
# STEP 1: SETUP & LIBRARIES
# ==============================================================================
import pandas as pd
import numpy as np
import os
import warnings
from typing import Dict, Any

warnings.filterwarnings('ignore')

try:
    import pyreadstat
except ImportError:
    print("Installing 'pyreadstat' package for stable data loading...")
    os.system('pip install pyreadstat')
    import pyreadstat

# ==============================================================================
# STEP 2: DATA INTEGRITY VERIFICATION SYSTEM
# ==============================================================================
def profile_dataset(df: pd.DataFrame, name: str) -> Dict[str, Any]:
    return {
        "name": name,
        "shape": df.shape,
        "columns": list(df.columns),
        "null_counts": df.isnull().sum().to_dict()
    }

def verify_data_integrity(df_original: pd.DataFrame, csv_path: str):
    print("\n" + "="*60)
    print("🔍 STEP 5: STARTING DATA INTEGRITY VERIFICATION")
    print("="*60)
    
    df_csv = pd.read_csv(csv_path)
    orig_prof = profile_dataset(df_original, "Processed DataFrame")
    csv_prof = profile_dataset(df_csv, "Saved CSV File")
    has_critical_error = False
    
    # Check 1: ตรวจสอบขนาดข้อมูล
    print("\n[Check 1/4] Checking Data Dimensions...")
    if orig_prof["shape"] == csv_prof["shape"]:
        print(f"✅ Success: Dimensions match perfectly! Shape: {orig_prof['shape']}")
    else:
        print(f"❌ CRITICAL DISCREPANCY: Dimensions do not match!")
        has_critical_error = True

    # Check 2: ตรวจสอบตำแหน่ง 2 คอลัมน์แรก (QID, hhid)
    print("\n[Check 2/4] Checking Identifier Column Positions...")
    first_two_cols = list(df_csv.columns)[:2]
    if first_two_cols == ['QID', 'hhid']:
        print(f"✅ Success: 'QID' and 'hhid' are safely locked at the first two columns! {first_two_cols}")
    else:
        print(f"❌ MISMATCH: Columns are not in the requested order. Found: {first_two_cols}")
        has_critical_error = True

    # Check 3: ตรวจสอบจำนวนแถว
    print("\n[Check 3/4] Checking Row Counts Consistency...")
    if len(df_original) == len(df_csv):
        print("✅ Success: Total Row count matches perfectly across both sources.")
    else:
        print("❌ MISMATCH: Row count discrepancy detected!")
        has_critical_error = True

    # Check 4: ตรวจสอบค่าความถูกต้องตัวเลข
    print("\n[Check 4/4] Checking Numeric Values Consistency...")
    try:
        numeric_cols = df_original.select_dtypes(include=[np.number]).columns.intersection(list(df_csv.columns))
        all_numeric_match = True
        for col in numeric_cols:
            o_val = df_original[col].fillna(-99999).to_numpy()
            c_val = df_csv[col].fillna(-99999).to_numpy()
            if not np.isclose(o_val, c_val, rtol=1e-05, atol=1e-08).all():
                print(f"⚠️ WARNING: Small numeric variation detected in column: [{col}]")
                all_numeric_match = False
        if all_numeric_match:
            print("✅ Success: Data values are completely secure and consistent.")
    except Exception as e:
        print(f"❌ Could not complete values check: {e}")
        has_critical_error = True

    print("\n" + "="*60)
    print("📊 VERIFICATION SUMMARY")
    print("="*60)
    if has_critical_error:
        print("🚨 CRITICAL ALERT: Data discrepancies found! Check the log details above.")
    else:
        print("🎉 PASSED: Data transformation is 100% complete, correct, and secure!")
    print("="*60)

# ==============================================================================
# STEP 3: DATA TRANSFORMATION WORKFLOW
# ==============================================================================
file_2017 = 'wave-1-2007-shocksclean.dta' 
output_file = 'wave-1-shocks_2007_raw_preserved.csv'

print("📥 STEP 2: Loading Stata file with PyReadStat...")
df_17, meta = pyreadstat.read_dta(file_2017)

def full_clean_wave7_2017_exact_preserved(df):
    print("🧹 STEP 3: Executing Harmonization & Splitting Coping Dummies by Exact Codes...")
    
    # 1. จัดการคอลัมน์ QID บ่งชี้ตัวตนครัวเรือน
    if 'QID' in df.columns:
        pass
    elif 'qid' in df.columns:
        df = df.rename(columns={'qid': 'QID'})
    elif 'hhid' in df.columns:
        df['QID'] = df['hhid']
    else:
        df['QID'] = np.nan

    # 2. ปรับเปลี่ยนรหัสระบุ Missing ทั่วไป (-9, -99) ให้เป็น NaN 
    df = df.replace([-9, -99, -9.0, -99.0], np.nan)
    
    # 3. รักษาธรรมชาติคอลัมน์เงิน (ชื่อเดิมแยกส่วน) ล้างคำว่า 'none' ออกเป็น 0
    money_cols = ['_x31005a', '_x31005b', '_x31006a']
    for col in money_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace('none', '0', case=False).str.strip()
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # 4. รายการรหัสกิจกรรมจริงครบถ้วนสมบูรณ์ตามที่คุณส่งมา 100% (ไม่รวมรหัสข้ามตัวเลือก 60, 61, 98)
    exact_coping_codes = [
        # กลุ่มเลขหลักเดียว (1-9 และ 10)
        1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
        # กลุ่มเลข 10 และ 20 (ขายทรัพย์สิน และการกู้ยืมเงิน)
        11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27,
        # กลุ่มเลข 30 (เงินช่วยเหลือและการบริจาค)
        28, 29, 30, 31,
        # กลุ่มเลข 40 (การปรับตัวทางอาชีพ/ธุรกิจ แรงงานและการเกษตร)
        40, 41, 42, 43,
        # กลุ่มเลขบริการพื้นฐานและการป้องกันภัย (48-55)
        48, 49, 50, 51, 52, 53, 54, 55,
        # กลุ่มกิจกรรมร่วมและอื่นๆ (62, 63, 90)
        62, 63, 90
    ]
    
    coping_source_cols = ['_x31008', '_x31009', '_x31010']
    df_coping_numeric = df[coping_source_cols].apply(pd.to_numeric, errors='coerce')
    
    print("   -> Splitting exact coping codes into wide dummy columns (0 or 1)...")
    for code in exact_coping_codes:
        # บันทึกเป็น 1 ทันทีหากเจอตัวเลขอ้างอิงนี้ในช่องความสำคัญใดความสำคัญหนึ่งจากทั้งสามช่องดั้งเดิม
        df[f'coping_{code}'] = df_coping_numeric.isin([code]).any(axis=1).astype(int)
        
    # ลอจิกป้องกันความผิดพลาดเชิงสถิติ: หากครัวเรือนระบุช่องแรกเป็น 98 (No Answer) ให้ Dummy ทั้งหมดในแถวนั้นเป็น NaN
    invalid_mask = df_coping_numeric['_x31008'].isin([98, 98.0])
    for code in exact_coping_codes:
        df.loc[invalid_mask, f'coping_{code}'] = np.nan

    # 5. คงค่าธรรมชาติคอลัมน์ระยะเวลาการฟื้นตัวดิบดั้งเดิมไว้ (_x31012a และ _x31012)
    if '_x31012a' in df.columns:
        df['_x31012a'] = pd.to_numeric(df['_x31012a'], errors='coerce')
        df['_x31012a'] = df['_x31012a'].replace([98, 99, 98.0, 99.0], np.nan)

    # 6. จัดกลุ่มประเภทภัยพิบัติภาพกว้าง (Shock Grouping) สำหรับสถิติพื้นฐาน
    df['survey_year'] = 2007
    if '_x31002' in df.columns:
        shock_mapping = {
            10: 'agricultural', 11: 'agricultural', 55: 'agricultural', 63: 'agricultural',
            1: 'demographic', 2: 'demographic', 3: 'demographic', 24: 'demographic',
            5: 'economics', 6: 'economics', 18: 'economics', 21: 'economics', 22: 'economics', 62: 'economics',
            8: 'social', 70: 'social', 77: 'economics'
        }
        shock_col = pd.to_numeric(df['_x31002'], errors='coerce')
        df['shocks_Group'] = shock_col.map(shock_mapping).fillna('others')

    # 7. ย้ายตำแหน่งคอลุมน์ระบุตัวตนหลัก (QID, hhid) มาล็อกไว้ที่ตำแหน่ง 2 คอลัมน์แรกสุดของตาราง
    remaining_cols = [col for col in df.columns if col not in ['QID', 'hhid']]
    ordered_cols = ['QID', 'hhid'] + remaining_cols
    df = df[ordered_cols]
    print("   -> Successfully rearranged columns: ['QID', 'hhid'] are now at the front.")

    return df

# เริ่มการรันกระบวนการแปลงข้อมูลทั้งหมด
df_17 = full_clean_wave7_2017_exact_preserved(df_17)

# ==============================================================================
# STEP 4: EXPORT & RUN INTEGRITY CHECK
# ==============================================================================
print(f"💾 STEP 4: Exporting preserved dataset to CSV: '{output_file}'...")
df_17.to_csv(output_file, index=False, encoding='utf-8-sig')

# ตรวจทาน Data Integrity อัตโนมัติหลังเซฟเสร็จ
verify_data_integrity(df_original=df_17, csv_path=output_file)

In [ ]:
# ==============================================================================
# STEP 1: SETUP & LIBRARIES
# ==============================================================================
import pandas as pd
import numpy as np
import os
import warnings
from typing import Dict, Any

warnings.filterwarnings('ignore')

try:
    import pyreadstat
except ImportError:
    print("Installing 'pyreadstat' package for stable data loading...")
    os.system('pip install pyreadstat')
    import pyreadstat

# ==============================================================================
# STEP 2: DATA INTEGRITY VERIFICATION SYSTEM
# ==============================================================================
def profile_dataset(df: pd.DataFrame, name: str) -> Dict[str, Any]:
    return {
        "name": name,
        "shape": df.shape,
        "columns": list(df.columns),
        "null_counts": df.isnull().sum().to_dict()
    }

def verify_data_integrity(df_original: pd.DataFrame, csv_path: str):
    print("\n" + "="*60)
    print("🔍 STEP 5: STARTING DATA INTEGRITY VERIFICATION")
    print("="*60)
    
    df_csv = pd.read_csv(csv_path)
    orig_prof = profile_dataset(df_original, "Processed DataFrame")
    csv_prof = profile_dataset(df_csv, "Saved CSV File")
    has_critical_error = False
    
    # Check 1: ตรวจสอบขนาดข้อมูล
    print("\n[Check 1/4] Checking Data Dimensions...")
    if orig_prof["shape"] == csv_prof["shape"]:
        print(f"✅ Success: Dimensions match perfectly! Shape: {orig_prof['shape']}")
    else:
        print(f"❌ CRITICAL DISCREPANCY: Dimensions do not match!")
        has_critical_error = True

    # Check 2: ตรวจสอบตำแหน่ง 2 คอลัมน์แรก (QID, hhid)
    print("\n[Check 2/4] Checking Identifier Column Positions...")
    first_two_cols = list(df_csv.columns)[:2]
    if first_two_cols == ['QID', 'hhid']:
        print(f"✅ Success: 'QID' and 'hhid' are safely locked at the first two columns! {first_two_cols}")
    else:
        print(f"❌ MISMATCH: Columns are not in the requested order. Found: {first_two_cols}")
        has_critical_error = True

    # Check 3: ตรวจสอบจำนวนแถว
    print("\n[Check 3/4] Checking Row Counts Consistency...")
    if len(df_original) == len(df_csv):
        print("✅ Success: Total Row count matches perfectly across both sources.")
    else:
        print("❌ MISMATCH: Row count discrepancy detected!")
        has_critical_error = True

    # Check 4: ตรวจสอบค่าความถูกต้องตัวเลข
    print("\n[Check 4/4] Checking Numeric Values Consistency...")
    try:
        numeric_cols = df_original.select_dtypes(include=[np.number]).columns.intersection(list(df_csv.columns))
        all_numeric_match = True
        for col in numeric_cols:
            o_val = df_original[col].fillna(-99999).to_numpy()
            c_val = df_csv[col].fillna(-99999).to_numpy()
            if not np.isclose(o_val, c_val, rtol=1e-05, atol=1e-08).all():
                print(f"⚠️ WARNING: Small numeric variation detected in column: [{col}]")
                all_numeric_match = False
        if all_numeric_match:
            print("✅ Success: Data values are completely secure and consistent.")
    except Exception as e:
        print(f"❌ Could not complete values check: {e}")
        has_critical_error = True

    print("\n" + "="*60)
    print("📊 VERIFICATION SUMMARY")
    print("="*60)
    if has_critical_error:
        print("🚨 CRITICAL ALERT: Data discrepancies found! Check the log details above.")
    else:
        print("🎉 PASSED: Data transformation is 100% complete, correct, and secure!")
    print("="*60)

# ==============================================================================
# STEP 3: DATA TRANSFORMATION WORKFLOW
# ==============================================================================
file_2017 = 'wave-2-2008-shocksclean.dta' 
output_file = 'wave-2-shocks_2008_raw_preserved.csv'

print("📥 STEP 2: Loading Stata file with PyReadStat...")
df_17, meta = pyreadstat.read_dta(file_2017)

def full_clean_wave7_2017_exact_preserved(df):
    print("🧹 STEP 3: Executing Harmonization & Splitting Coping Dummies by Exact Codes...")
    
    # 1. จัดการคอลัมน์ QID บ่งชี้ตัวตนครัวเรือน
    if 'QID' in df.columns:
        pass
    elif 'qid' in df.columns:
        df = df.rename(columns={'qid': 'QID'})
    elif 'hhid' in df.columns:
        df['QID'] = df['hhid']
    else:
        df['QID'] = np.nan

    # 2. ปรับเปลี่ยนรหัสระบุ Missing ทั่วไป (-9, -99) ให้เป็น NaN 
    df = df.replace([-9, -99, -9.0, -99.0], np.nan)
    
    # 3. รักษาธรรมชาติคอลัมน์เงิน (ชื่อเดิมแยกส่วน) ล้างคำว่า 'none' ออกเป็น 0
    money_cols = ['_x31005a', '_x31005b', '_x31006a']
    for col in money_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace('none', '0', case=False).str.strip()
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # 4. รายการรหัสกิจกรรมจริงครบถ้วนสมบูรณ์ตามที่คุณส่งมา 100% (ไม่รวมรหัสข้ามตัวเลือก 60, 61, 98)
    exact_coping_codes = [
        # กลุ่มเลขหลักเดียว (1-9 และ 10)
        1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
        # กลุ่มเลข 10 และ 20 (ขายทรัพย์สิน และการกู้ยืมเงิน)
        11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27,
        # กลุ่มเลข 30 (เงินช่วยเหลือและการบริจาค)
        28, 29, 30, 31,
        # กลุ่มเลข 40 (การปรับตัวทางอาชีพ/ธุรกิจ แรงงานและการเกษตร)
        40, 41, 42, 43,
        # กลุ่มเลขบริการพื้นฐานและการป้องกันภัย (48-55)
        48, 49, 50, 51, 52, 53, 54, 55,
        # กลุ่มกิจกรรมร่วมและอื่นๆ (62, 63, 90)
        62, 63, 90
    ]
    
    coping_source_cols = ['_x31008', '_x31009', '_x31010']
    df_coping_numeric = df[coping_source_cols].apply(pd.to_numeric, errors='coerce')
    
    print("   -> Splitting exact coping codes into wide dummy columns (0 or 1)...")
    for code in exact_coping_codes:
        # บันทึกเป็น 1 ทันทีหากเจอตัวเลขอ้างอิงนี้ในช่องความสำคัญใดความสำคัญหนึ่งจากทั้งสามช่องดั้งเดิม
        df[f'coping_{code}'] = df_coping_numeric.isin([code]).any(axis=1).astype(int)
        
    # ลอจิกป้องกันความผิดพลาดเชิงสถิติ: หากครัวเรือนระบุช่องแรกเป็น 98 (No Answer) ให้ Dummy ทั้งหมดในแถวนั้นเป็น NaN
    invalid_mask = df_coping_numeric['_x31008'].isin([98, 98.0])
    for code in exact_coping_codes:
        df.loc[invalid_mask, f'coping_{code}'] = np.nan

    # 5. คงค่าธรรมชาติคอลัมน์ระยะเวลาการฟื้นตัวดิบดั้งเดิมไว้ (_x31012a และ _x31012)
    if '_x31012a' in df.columns:
        df['_x31012a'] = pd.to_numeric(df['_x31012a'], errors='coerce')
        df['_x31012a'] = df['_x31012a'].replace([98, 99, 98.0, 99.0], np.nan)

    # 6. จัดกลุ่มประเภทภัยพิบัติภาพกว้าง (Shock Grouping) สำหรับสถิติพื้นฐาน
    df['survey_year'] = 2008
    if '_x31002' in df.columns:
        shock_mapping = {
            10: 'agricultural', 11: 'agricultural', 55: 'agricultural', 63: 'agricultural',
            1: 'demographic', 2: 'demographic', 3: 'demographic', 24: 'demographic',
            5: 'economics', 6: 'economics', 18: 'economics', 21: 'economics', 22: 'economics', 62: 'economics',
            8: 'social', 70: 'social', 77: 'economics'
        }
        shock_col = pd.to_numeric(df['_x31002'], errors='coerce')
        df['shocks_Group'] = shock_col.map(shock_mapping).fillna('others')

    # 7. ย้ายตำแหน่งคอลุมน์ระบุตัวตนหลัก (QID, hhid) มาล็อกไว้ที่ตำแหน่ง 2 คอลัมน์แรกสุดของตาราง
    remaining_cols = [col for col in df.columns if col not in ['QID', 'hhid']]
    ordered_cols = ['QID', 'hhid'] + remaining_cols
    df = df[ordered_cols]
    print("   -> Successfully rearranged columns: ['QID', 'hhid'] are now at the front.")

    return df

# เริ่มการรันกระบวนการแปลงข้อมูลทั้งหมด
df_17 = full_clean_wave7_2017_exact_preserved(df_17)

# ==============================================================================
# STEP 4: EXPORT & RUN INTEGRITY CHECK
# ==============================================================================
print(f"💾 STEP 4: Exporting preserved dataset to CSV: '{output_file}'...")
df_17.to_csv(output_file, index=False, encoding='utf-8-sig')

# ตรวจทาน Data Integrity อัตโนมัติหลังเซฟเสร็จ
verify_data_integrity(df_original=df_17, csv_path=output_file)

In [ ]:
# ==============================================================================
# STEP 1: SETUP & LIBRARIES
# ==============================================================================
import pandas as pd
import numpy as np
import os
import warnings
from typing import Dict, Any

warnings.filterwarnings('ignore')

try:
    import pyreadstat
except ImportError:
    print("Installing 'pyreadstat' package for stable data loading...")
    os.system('pip install pyreadstat')
    import pyreadstat

# ==============================================================================
# STEP 2: DATA INTEGRITY VERIFICATION SYSTEM
# ==============================================================================
def profile_dataset(df: pd.DataFrame, name: str) -> Dict[str, Any]:
    return {
        "name": name,
        "shape": df.shape,
        "columns": list(df.columns),
        "null_counts": df.isnull().sum().to_dict()
    }

def verify_data_integrity(df_original: pd.DataFrame, csv_path: str):
    print("\n" + "="*60)
    print("🔍 STEP 5: STARTING DATA INTEGRITY VERIFICATION")
    print("="*60)
    
    df_csv = pd.read_csv(csv_path)
    orig_prof = profile_dataset(df_original, "Processed DataFrame")
    csv_prof = profile_dataset(df_csv, "Saved CSV File")
    has_critical_error = False
    
    # Check 1: ตรวจสอบขนาดข้อมูล
    print("\n[Check 1/4] Checking Data Dimensions...")
    if orig_prof["shape"] == csv_prof["shape"]:
        print(f"✅ Success: Dimensions match perfectly! Shape: {orig_prof['shape']}")
    else:
        print(f"❌ CRITICAL DISCREPANCY: Dimensions do not match!")
        has_critical_error = True

    # Check 2: ตรวจสอบตำแหน่ง 2 คอลัมน์แรก (QID, hhid)
    print("\n[Check 2/4] Checking Identifier Column Positions...")
    first_two_cols = list(df_csv.columns)[:2]
    if first_two_cols == ['QID', 'hhid']:
        print(f"✅ Success: 'QID' and 'hhid' are safely locked at the first two columns! {first_two_cols}")
    else:
        print(f"❌ MISMATCH: Columns are not in the requested order. Found: {first_two_cols}")
        has_critical_error = True

    # Check 3: ตรวจสอบจำนวนแถว
    print("\n[Check 3/4] Checking Row Counts Consistency...")
    if len(df_original) == len(df_csv):
        print("✅ Success: Total Row count matches perfectly across both sources.")
    else:
        print("❌ MISMATCH: Row count discrepancy detected!")
        has_critical_error = True

    # Check 4: ตรวจสอบค่าความถูกต้องตัวเลข
    print("\n[Check 4/4] Checking Numeric Values Consistency...")
    try:
        numeric_cols = df_original.select_dtypes(include=[np.number]).columns.intersection(list(df_csv.columns))
        all_numeric_match = True
        for col in numeric_cols:
            o_val = df_original[col].fillna(-99999).to_numpy()
            c_val = df_csv[col].fillna(-99999).to_numpy()
            if not np.isclose(o_val, c_val, rtol=1e-05, atol=1e-08).all():
                print(f"⚠️ WARNING: Small numeric variation detected in column: [{col}]")
                all_numeric_match = False
        if all_numeric_match:
            print("✅ Success: Data values are completely secure and consistent.")
    except Exception as e:
        print(f"❌ Could not complete values check: {e}")
        has_critical_error = True

    print("\n" + "="*60)
    print("📊 VERIFICATION SUMMARY")
    print("="*60)
    if has_critical_error:
        print("🚨 CRITICAL ALERT: Data discrepancies found! Check the log details above.")
    else:
        print("🎉 PASSED: Data transformation is 100% complete, correct, and secure!")
    print("="*60)

# ==============================================================================
# STEP 3: DATA TRANSFORMATION WORKFLOW
# ==============================================================================
file_2017 = 'wave-3-2010-shocksclean.dta' 
output_file = 'wave-3-shocks_2010_raw_preserved.csv'

print("📥 STEP 2: Loading Stata file with PyReadStat...")
df_17, meta = pyreadstat.read_dta(file_2017)

def full_clean_wave7_2017_exact_preserved(df):
    print("🧹 STEP 3: Executing Harmonization & Splitting Coping Dummies by Exact Codes...")
    
    # 1. จัดการคอลัมน์ QID บ่งชี้ตัวตนครัวเรือน
    if 'QID' in df.columns:
        pass
    elif 'qid' in df.columns:
        df = df.rename(columns={'qid': 'QID'})
    elif 'hhid' in df.columns:
        df['QID'] = df['hhid']
    else:
        df['QID'] = np.nan

    # 2. ปรับเปลี่ยนรหัสระบุ Missing ทั่วไป (-9, -99) ให้เป็น NaN 
    df = df.replace([-9, -99, -9.0, -99.0], np.nan)
    
    # 3. รักษาธรรมชาติคอลัมน์เงิน (ชื่อเดิมแยกส่วน) ล้างคำว่า 'none' ออกเป็น 0
    money_cols = ['_x31005a', '_x31005b', '_x31006a']
    for col in money_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace('none', '0', case=False).str.strip()
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # 4. รายการรหัสกิจกรรมจริงครบถ้วนสมบูรณ์ตามที่คุณส่งมา 100% (ไม่รวมรหัสข้ามตัวเลือก 60, 61, 98)
    exact_coping_codes = [
        # กลุ่มเลขหลักเดียว (1-9 และ 10)
        1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
        # กลุ่มเลข 10 และ 20 (ขายทรัพย์สิน และการกู้ยืมเงิน)
        11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27,
        # กลุ่มเลข 30 (เงินช่วยเหลือและการบริจาค)
        28, 29, 30, 31,
        # กลุ่มเลข 40 (การปรับตัวทางอาชีพ/ธุรกิจ แรงงานและการเกษตร)
        40, 41, 42, 43,
        # กลุ่มเลขบริการพื้นฐานและการป้องกันภัย (48-55)
        48, 49, 50, 51, 52, 53, 54, 55,
        # กลุ่มกิจกรรมร่วมและอื่นๆ (62, 63, 90)
        62, 63, 90
    ]
    
    coping_source_cols = ['_x31008', '_x31009', '_x31010']
    df_coping_numeric = df[coping_source_cols].apply(pd.to_numeric, errors='coerce')
    
    print("   -> Splitting exact coping codes into wide dummy columns (0 or 1)...")
    for code in exact_coping_codes:
        # บันทึกเป็น 1 ทันทีหากเจอตัวเลขอ้างอิงนี้ในช่องความสำคัญใดความสำคัญหนึ่งจากทั้งสามช่องดั้งเดิม
        df[f'coping_{code}'] = df_coping_numeric.isin([code]).any(axis=1).astype(int)
        
    # ลอจิกป้องกันความผิดพลาดเชิงสถิติ: หากครัวเรือนระบุช่องแรกเป็น 98 (No Answer) ให้ Dummy ทั้งหมดในแถวนั้นเป็น NaN
    invalid_mask = df_coping_numeric['_x31008'].isin([98, 98.0])
    for code in exact_coping_codes:
        df.loc[invalid_mask, f'coping_{code}'] = np.nan

    # 5. คงค่าธรรมชาติคอลัมน์ระยะเวลาการฟื้นตัวดิบดั้งเดิมไว้ (_x31012a และ _x31012)
    if '_x31012a' in df.columns:
        df['_x31012a'] = pd.to_numeric(df['_x31012a'], errors='coerce')
        df['_x31012a'] = df['_x31012a'].replace([98, 99, 98.0, 99.0], np.nan)

    # 6. จัดกลุ่มประเภทภัยพิบัติภาพกว้าง (Shock Grouping) สำหรับสถิติพื้นฐาน
    df['survey_year'] = 2010
    if '_x31002' in df.columns:
        shock_mapping = {
            10: 'agricultural', 11: 'agricultural', 55: 'agricultural', 63: 'agricultural',
            1: 'demographic', 2: 'demographic', 3: 'demographic', 24: 'demographic',
            5: 'economics', 6: 'economics', 18: 'economics', 21: 'economics', 22: 'economics', 62: 'economics',
            8: 'social', 70: 'social', 77: 'economics'
        }
        shock_col = pd.to_numeric(df['_x31002'], errors='coerce')
        df['shocks_Group'] = shock_col.map(shock_mapping).fillna('others')

    # 7. ย้ายตำแหน่งคอลุมน์ระบุตัวตนหลัก (QID, hhid) มาล็อกไว้ที่ตำแหน่ง 2 คอลัมน์แรกสุดของตาราง
    remaining_cols = [col for col in df.columns if col not in ['QID', 'hhid']]
    ordered_cols = ['QID', 'hhid'] + remaining_cols
    df = df[ordered_cols]
    print("   -> Successfully rearranged columns: ['QID', 'hhid'] are now at the front.")

    return df

# เริ่มการรันกระบวนการแปลงข้อมูลทั้งหมด
df_17 = full_clean_wave7_2017_exact_preserved(df_17)

# ==============================================================================
# STEP 4: EXPORT & RUN INTEGRITY CHECK
# ==============================================================================
print(f"💾 STEP 4: Exporting preserved dataset to CSV: '{output_file}'...")
df_17.to_csv(output_file, index=False, encoding='utf-8-sig')

# ตรวจทาน Data Integrity อัตโนมัติหลังเซฟเสร็จ
verify_data_integrity(df_original=df_17, csv_path=output_file)

In [ ]:
# ==============================================================================
# STEP 1: SETUP & LIBRARIES
# ==============================================================================
import pandas as pd
import numpy as np
import os
import warnings
from typing import Dict, Any

warnings.filterwarnings('ignore')

try:
    import pyreadstat
except ImportError:
    print("Installing 'pyreadstat' package for stable data loading...")
    os.system('pip install pyreadstat')
    import pyreadstat

# ==============================================================================
# STEP 2: DATA INTEGRITY VERIFICATION SYSTEM
# ==============================================================================
def profile_dataset(df: pd.DataFrame, name: str) -> Dict[str, Any]:
    return {
        "name": name,
        "shape": df.shape,
        "columns": list(df.columns),
        "null_counts": df.isnull().sum().to_dict()
    }

def verify_data_integrity(df_original: pd.DataFrame, csv_path: str):
    print("\n" + "="*60)
    print("🔍 STEP 5: STARTING DATA INTEGRITY VERIFICATION")
    print("="*60)
    
    df_csv = pd.read_csv(csv_path)
    orig_prof = profile_dataset(df_original, "Processed DataFrame")
    csv_prof = profile_dataset(df_csv, "Saved CSV File")
    has_critical_error = False
    
    # Check 1: ตรวจสอบขนาดข้อมูล
    print("\n[Check 1/4] Checking Data Dimensions...")
    if orig_prof["shape"] == csv_prof["shape"]:
        print(f"✅ Success: Dimensions match perfectly! Shape: {orig_prof['shape']}")
    else:
        print(f"❌ CRITICAL DISCREPANCY: Dimensions do not match!")
        has_critical_error = True

    # Check 2: ตรวจสอบตำแหน่ง 2 คอลัมน์แรก (QID, hhid)
    print("\n[Check 2/4] Checking Identifier Column Positions...")
    first_two_cols = list(df_csv.columns)[:2]
    if first_two_cols == ['QID', 'hhid']:
        print(f"✅ Success: 'QID' and 'hhid' are safely locked at the first two columns! {first_two_cols}")
    else:
        print(f"❌ MISMATCH: Columns are not in the requested order. Found: {first_two_cols}")
        has_critical_error = True

    # Check 3: ตรวจสอบจำนวนแถว
    print("\n[Check 3/4] Checking Row Counts Consistency...")
    if len(df_original) == len(df_csv):
        print("✅ Success: Total Row count matches perfectly across both sources.")
    else:
        print("❌ MISMATCH: Row count discrepancy detected!")
        has_critical_error = True

    # Check 4: ตรวจสอบค่าความถูกต้องตัวเลข
    print("\n[Check 4/4] Checking Numeric Values Consistency...")
    try:
        numeric_cols = df_original.select_dtypes(include=[np.number]).columns.intersection(list(df_csv.columns))
        all_numeric_match = True
        for col in numeric_cols:
            o_val = df_original[col].fillna(-99999).to_numpy()
            c_val = df_csv[col].fillna(-99999).to_numpy()
            if not np.isclose(o_val, c_val, rtol=1e-05, atol=1e-08).all():
                print(f"⚠️ WARNING: Small numeric variation detected in column: [{col}]")
                all_numeric_match = False
        if all_numeric_match:
            print("✅ Success: Data values are completely secure and consistent.")
    except Exception as e:
        print(f"❌ Could not complete values check: {e}")
        has_critical_error = True

    print("\n" + "="*60)
    print("📊 VERIFICATION SUMMARY")
    print("="*60)
    if has_critical_error:
        print("🚨 CRITICAL ALERT: Data discrepancies found! Check the log details above.")
    else:
        print("🎉 PASSED: Data transformation is 100% complete, correct, and secure!")
    print("="*60)

# ==============================================================================
# STEP 3: DATA TRANSFORMATION WORKFLOW
# ==============================================================================
file_2017 = 'wave-4-2011-shocksclean.dta' 
output_file = 'wave-4-shocks_2011_raw_preserved.csv'

print("📥 STEP 2: Loading Stata file with PyReadStat...")
df_17, meta = pyreadstat.read_dta(file_2017)

def full_clean_wave7_2017_exact_preserved(df):
    print("🧹 STEP 3: Executing Harmonization & Splitting Coping Dummies by Exact Codes...")
    
    # 1. จัดการคอลัมน์ QID บ่งชี้ตัวตนครัวเรือน
    if 'QID' in df.columns:
        pass
    elif 'qid' in df.columns:
        df = df.rename(columns={'qid': 'QID'})
    elif 'hhid' in df.columns:
        df['QID'] = df['hhid']
    else:
        df['QID'] = np.nan

    # 2. ปรับเปลี่ยนรหัสระบุ Missing ทั่วไป (-9, -99) ให้เป็น NaN 
    df = df.replace([-9, -99, -9.0, -99.0], np.nan)
    
    # 3. รักษาธรรมชาติคอลัมน์เงิน (ชื่อเดิมแยกส่วน) ล้างคำว่า 'none' ออกเป็น 0
    money_cols = ['_x31005a', '_x31005b', '_x31006a']
    for col in money_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace('none', '0', case=False).str.strip()
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # 4. รายการรหัสกิจกรรมจริงครบถ้วนสมบูรณ์ตามที่คุณส่งมา 100% (ไม่รวมรหัสข้ามตัวเลือก 60, 61, 98)
    exact_coping_codes = [
        # กลุ่มเลขหลักเดียว (1-9 และ 10)
        1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
        # กลุ่มเลข 10 และ 20 (ขายทรัพย์สิน และการกู้ยืมเงิน)
        11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27,
        # กลุ่มเลข 30 (เงินช่วยเหลือและการบริจาค)
        28, 29, 30, 31,
        # กลุ่มเลข 40 (การปรับตัวทางอาชีพ/ธุรกิจ แรงงานและการเกษตร)
        40, 41, 42, 43,
        # กลุ่มเลขบริการพื้นฐานและการป้องกันภัย (48-55)
        48, 49, 50, 51, 52, 53, 54, 55,
        # กลุ่มกิจกรรมร่วมและอื่นๆ (62, 63, 90)
        62, 63, 90
    ]
    
    coping_source_cols = ['_x31008', '_x31009', '_x31010']
    df_coping_numeric = df[coping_source_cols].apply(pd.to_numeric, errors='coerce')
    
    print("   -> Splitting exact coping codes into wide dummy columns (0 or 1)...")
    for code in exact_coping_codes:
        # บันทึกเป็น 1 ทันทีหากเจอตัวเลขอ้างอิงนี้ในช่องความสำคัญใดความสำคัญหนึ่งจากทั้งสามช่องดั้งเดิม
        df[f'coping_{code}'] = df_coping_numeric.isin([code]).any(axis=1).astype(int)
        
    # ลอจิกป้องกันความผิดพลาดเชิงสถิติ: หากครัวเรือนระบุช่องแรกเป็น 98 (No Answer) ให้ Dummy ทั้งหมดในแถวนั้นเป็น NaN
    invalid_mask = df_coping_numeric['_x31008'].isin([98, 98.0])
    for code in exact_coping_codes:
        df.loc[invalid_mask, f'coping_{code}'] = np.nan

    # 5. คงค่าธรรมชาติคอลัมน์ระยะเวลาการฟื้นตัวดิบดั้งเดิมไว้ (_x31012a และ _x31012)
    if '_x31012a' in df.columns:
        df['_x31012a'] = pd.to_numeric(df['_x31012a'], errors='coerce')
        df['_x31012a'] = df['_x31012a'].replace([98, 99, 98.0, 99.0], np.nan)

    # 6. จัดกลุ่มประเภทภัยพิบัติภาพกว้าง (Shock Grouping) สำหรับสถิติพื้นฐาน
    df['survey_year'] = 2011
    if '_x31002' in df.columns:
        shock_mapping = {
            10: 'agricultural', 11: 'agricultural', 55: 'agricultural', 63: 'agricultural',
            1: 'demographic', 2: 'demographic', 3: 'demographic', 24: 'demographic',
            5: 'economics', 6: 'economics', 18: 'economics', 21: 'economics', 22: 'economics', 62: 'economics',
            8: 'social', 70: 'social', 77: 'economics'
        }
        shock_col = pd.to_numeric(df['_x31002'], errors='coerce')
        df['shocks_Group'] = shock_col.map(shock_mapping).fillna('others')

    # 7. ย้ายตำแหน่งคอลุมน์ระบุตัวตนหลัก (QID, hhid) มาล็อกไว้ที่ตำแหน่ง 2 คอลัมน์แรกสุดของตาราง
    remaining_cols = [col for col in df.columns if col not in ['QID', 'hhid']]
    ordered_cols = ['QID', 'hhid'] + remaining_cols
    df = df[ordered_cols]
    print("   -> Successfully rearranged columns: ['QID', 'hhid'] are now at the front.")

    return df

# เริ่มการรันกระบวนการแปลงข้อมูลทั้งหมด
df_17 = full_clean_wave7_2017_exact_preserved(df_17)

# ==============================================================================
# STEP 4: EXPORT & RUN INTEGRITY CHECK
# ==============================================================================
print(f"💾 STEP 4: Exporting preserved dataset to CSV: '{output_file}'...")
df_17.to_csv(output_file, index=False, encoding='utf-8-sig')

# ตรวจทาน Data Integrity อัตโนมัติหลังเซฟเสร็จ
verify_data_integrity(df_original=df_17, csv_path=output_file)

In [ ]:
# ==============================================================================
# STEP 1: SETUP & LIBRARIES
# ==============================================================================
import pandas as pd
import numpy as np
import os
import warnings
from typing import Dict, Any

warnings.filterwarnings('ignore')

try:
    import pyreadstat
except ImportError:
    print("Installing 'pyreadstat' package for stable data loading...")
    os.system('pip install pyreadstat')
    import pyreadstat

# ==============================================================================
# STEP 2: DATA INTEGRITY VERIFICATION SYSTEM
# ==============================================================================
def profile_dataset(df: pd.DataFrame, name: str) -> Dict[str, Any]:
    return {
        "name": name,
        "shape": df.shape,
        "columns": list(df.columns),
        "null_counts": df.isnull().sum().to_dict()
    }

def verify_data_integrity(df_original: pd.DataFrame, csv_path: str):
    print("\n" + "="*60)
    print("🔍 STEP 5: STARTING DATA INTEGRITY VERIFICATION")
    print("="*60)
    
    df_csv = pd.read_csv(csv_path)
    orig_prof = profile_dataset(df_original, "Processed DataFrame")
    csv_prof = profile_dataset(df_csv, "Saved CSV File")
    has_critical_error = False
    
    # Check 1: ตรวจสอบขนาดข้อมูล
    print("\n[Check 1/4] Checking Data Dimensions...")
    if orig_prof["shape"] == csv_prof["shape"]:
        print(f"✅ Success: Dimensions match perfectly! Shape: {orig_prof['shape']}")
    else:
        print(f"❌ CRITICAL DISCREPANCY: Dimensions do not match!")
        has_critical_error = True

    # Check 2: ตรวจสอบตำแหน่ง 2 คอลัมน์แรก (QID, hhid)
    print("\n[Check 2/4] Checking Identifier Column Positions...")
    first_two_cols = list(df_csv.columns)[:2]
    if first_two_cols == ['QID', 'hhid']:
        print(f"✅ Success: 'QID' and 'hhid' are safely locked at the first two columns! {first_two_cols}")
    else:
        print(f"❌ MISMATCH: Columns are not in the requested order. Found: {first_two_cols}")
        has_critical_error = True

    # Check 3: ตรวจสอบจำนวนแถว
    print("\n[Check 3/4] Checking Row Counts Consistency...")
    if len(df_original) == len(df_csv):
        print("✅ Success: Total Row count matches perfectly across both sources.")
    else:
        print("❌ MISMATCH: Row count discrepancy detected!")
        has_critical_error = True

    # Check 4: ตรวจสอบค่าความถูกต้องตัวเลข
    print("\n[Check 4/4] Checking Numeric Values Consistency...")
    try:
        numeric_cols = df_original.select_dtypes(include=[np.number]).columns.intersection(list(df_csv.columns))
        all_numeric_match = True
        for col in numeric_cols:
            o_val = df_original[col].fillna(-99999).to_numpy()
            c_val = df_csv[col].fillna(-99999).to_numpy()
            if not np.isclose(o_val, c_val, rtol=1e-05, atol=1e-08).all():
                print(f"⚠️ WARNING: Small numeric variation detected in column: [{col}]")
                all_numeric_match = False
        if all_numeric_match:
            print("✅ Success: Data values are completely secure and consistent.")
    except Exception as e:
        print(f"❌ Could not complete values check: {e}")
        has_critical_error = True

    print("\n" + "="*60)
    print("📊 VERIFICATION SUMMARY")
    print("="*60)
    if has_critical_error:
        print("🚨 CRITICAL ALERT: Data discrepancies found! Check the log details above.")
    else:
        print("🎉 PASSED: Data transformation is 100% complete, correct, and secure!")
    print("="*60)

# ==============================================================================
# STEP 3: DATA TRANSFORMATION WORKFLOW
# ==============================================================================
file_2017 = 'wave-5-2013-shocksclean.dta' 
output_file = 'wave-5-shocks_2013_raw_preserved.csv'

print("📥 STEP 2: Loading Stata file with PyReadStat...")
df_17, meta = pyreadstat.read_dta(file_2017)

def full_clean_wave7_2017_exact_preserved(df):
    print("🧹 STEP 3: Executing Harmonization & Splitting Coping Dummies by Exact Codes...")
    
    # 1. จัดการคอลัมน์ QID บ่งชี้ตัวตนครัวเรือน
    if 'QID' in df.columns:
        pass
    elif 'qid' in df.columns:
        df = df.rename(columns={'qid': 'QID'})
    elif 'hhid' in df.columns:
        df['QID'] = df['hhid']
    else:
        df['QID'] = np.nan

    # 2. ปรับเปลี่ยนรหัสระบุ Missing ทั่วไป (-9, -99) ให้เป็น NaN 
    df = df.replace([-9, -99, -9.0, -99.0], np.nan)
    
    # 3. รักษาธรรมชาติคอลัมน์เงิน (ชื่อเดิมแยกส่วน) ล้างคำว่า 'none' ออกเป็น 0
    money_cols = ['_x31005a', '_x31005b', '_x31006a']
    for col in money_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace('none', '0', case=False).str.strip()
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # 4. รายการรหัสกิจกรรมจริงครบถ้วนสมบูรณ์ตามที่คุณส่งมา 100% (ไม่รวมรหัสข้ามตัวเลือก 60, 61, 98)
    exact_coping_codes = [
        # กลุ่มเลขหลักเดียว (1-9 และ 10)
        1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
        # กลุ่มเลข 10 และ 20 (ขายทรัพย์สิน และการกู้ยืมเงิน)
        11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27,
        # กลุ่มเลข 30 (เงินช่วยเหลือและการบริจาค)
        28, 29, 30, 31,
        # กลุ่มเลข 40 (การปรับตัวทางอาชีพ/ธุรกิจ แรงงานและการเกษตร)
        40, 41, 42, 43,
        # กลุ่มเลขบริการพื้นฐานและการป้องกันภัย (48-55)
        48, 49, 50, 51, 52, 53, 54, 55,
        # กลุ่มกิจกรรมร่วมและอื่นๆ (62, 63, 90)
        62, 63, 90
    ]
    
    coping_source_cols = ['_x31008', '_x31009', '_x31010']
    df_coping_numeric = df[coping_source_cols].apply(pd.to_numeric, errors='coerce')
    
    print("   -> Splitting exact coping codes into wide dummy columns (0 or 1)...")
    for code in exact_coping_codes:
        # บันทึกเป็น 1 ทันทีหากเจอตัวเลขอ้างอิงนี้ในช่องความสำคัญใดความสำคัญหนึ่งจากทั้งสามช่องดั้งเดิม
        df[f'coping_{code}'] = df_coping_numeric.isin([code]).any(axis=1).astype(int)
        
    # ลอจิกป้องกันความผิดพลาดเชิงสถิติ: หากครัวเรือนระบุช่องแรกเป็น 98 (No Answer) ให้ Dummy ทั้งหมดในแถวนั้นเป็น NaN
    invalid_mask = df_coping_numeric['_x31008'].isin([98, 98.0])
    for code in exact_coping_codes:
        df.loc[invalid_mask, f'coping_{code}'] = np.nan

    # 5. คงค่าธรรมชาติคอลัมน์ระยะเวลาการฟื้นตัวดิบดั้งเดิมไว้ (_x31012a และ _x31012)
    if '_x31012a' in df.columns:
        df['_x31012a'] = pd.to_numeric(df['_x31012a'], errors='coerce')
        df['_x31012a'] = df['_x31012a'].replace([98, 99, 98.0, 99.0], np.nan)

    # 6. จัดกลุ่มประเภทภัยพิบัติภาพกว้าง (Shock Grouping) สำหรับสถิติพื้นฐาน
    df['survey_year'] = 2013
    if '_x31002' in df.columns:
        shock_mapping = {
            10: 'agricultural', 11: 'agricultural', 55: 'agricultural', 63: 'agricultural',
            1: 'demographic', 2: 'demographic', 3: 'demographic', 24: 'demographic',
            5: 'economics', 6: 'economics', 18: 'economics', 21: 'economics', 22: 'economics', 62: 'economics',
            8: 'social', 70: 'social', 77: 'economics'
        }
        shock_col = pd.to_numeric(df['_x31002'], errors='coerce')
        df['shocks_Group'] = shock_col.map(shock_mapping).fillna('others')

    # 7. ย้ายตำแหน่งคอลุมน์ระบุตัวตนหลัก (QID, hhid) มาล็อกไว้ที่ตำแหน่ง 2 คอลัมน์แรกสุดของตาราง
    remaining_cols = [col for col in df.columns if col not in ['QID', 'hhid']]
    ordered_cols = ['QID', 'hhid'] + remaining_cols
    df = df[ordered_cols]
    print("   -> Successfully rearranged columns: ['QID', 'hhid'] are now at the front.")

    return df

# เริ่มการรันกระบวนการแปลงข้อมูลทั้งหมด
df_17 = full_clean_wave7_2017_exact_preserved(df_17)

# ==============================================================================
# STEP 4: EXPORT & RUN INTEGRITY CHECK
# ==============================================================================
print(f"💾 STEP 4: Exporting preserved dataset to CSV: '{output_file}'...")
df_17.to_csv(output_file, index=False, encoding='utf-8-sig')

# ตรวจทาน Data Integrity อัตโนมัติหลังเซฟเสร็จ
verify_data_integrity(df_original=df_17, csv_path=output_file)

In [ ]:
# ==============================================================================
# STEP 1: SETUP & LIBRARIES
# ==============================================================================
import pandas as pd
import numpy as np
import os
import warnings
from typing import Dict, Any

warnings.filterwarnings('ignore')

try:
    import pyreadstat
except ImportError:
    print("Installing 'pyreadstat' package for stable data loading...")
    os.system('pip install pyreadstat')
    import pyreadstat

# ==============================================================================
# STEP 2: DATA INTEGRITY VERIFICATION SYSTEM
# ==============================================================================
def profile_dataset(df: pd.DataFrame, name: str) -> Dict[str, Any]:
    return {
        "name": name,
        "shape": df.shape,
        "columns": list(df.columns),
        "null_counts": df.isnull().sum().to_dict()
    }

def verify_data_integrity(df_original: pd.DataFrame, csv_path: str):
    print("\n" + "="*60)
    print("🔍 STEP 5: STARTING DATA INTEGRITY VERIFICATION")
    print("="*60)
    
    df_csv = pd.read_csv(csv_path)
    orig_prof = profile_dataset(df_original, "Processed DataFrame")
    csv_prof = profile_dataset(df_csv, "Saved CSV File")
    has_critical_error = False
    
    # Check 1: ตรวจสอบขนาดข้อมูล
    print("\n[Check 1/4] Checking Data Dimensions...")
    if orig_prof["shape"] == csv_prof["shape"]:
        print(f"✅ Success: Dimensions match perfectly! Shape: {orig_prof['shape']}")
    else:
        print(f"❌ CRITICAL DISCREPANCY: Dimensions do not match!")
        has_critical_error = True

    # Check 2: ตรวจสอบตำแหน่ง 2 คอลัมน์แรก (QID, hhid)
    print("\n[Check 2/4] Checking Identifier Column Positions...")
    first_two_cols = list(df_csv.columns)[:2]
    if first_two_cols == ['QID', 'hhid']:
        print(f"✅ Success: 'QID' and 'hhid' are safely locked at the first two columns! {first_two_cols}")
    else:
        print(f"❌ MISMATCH: Columns are not in the requested order. Found: {first_two_cols}")
        has_critical_error = True

    # Check 3: ตรวจสอบจำนวนแถว
    print("\n[Check 3/4] Checking Row Counts Consistency...")
    if len(df_original) == len(df_csv):
        print("✅ Success: Total Row count matches perfectly across both sources.")
    else:
        print("❌ MISMATCH: Row count discrepancy detected!")
        has_critical_error = True

    # Check 4: ตรวจสอบค่าความถูกต้องตัวเลข
    print("\n[Check 4/4] Checking Numeric Values Consistency...")
    try:
        numeric_cols = df_original.select_dtypes(include=[np.number]).columns.intersection(list(df_csv.columns))
        all_numeric_match = True
        for col in numeric_cols:
            o_val = df_original[col].fillna(-99999).to_numpy()
            c_val = df_csv[col].fillna(-99999).to_numpy()
            if not np.isclose(o_val, c_val, rtol=1e-05, atol=1e-08).all():
                print(f"⚠️ WARNING: Small numeric variation detected in column: [{col}]")
                all_numeric_match = False
        if all_numeric_match:
            print("✅ Success: Data values are completely secure and consistent.")
    except Exception as e:
        print(f"❌ Could not complete values check: {e}")
        has_critical_error = True

    print("\n" + "="*60)
    print("📊 VERIFICATION SUMMARY")
    print("="*60)
    if has_critical_error:
        print("🚨 CRITICAL ALERT: Data discrepancies found! Check the log details above.")
    else:
        print("🎉 PASSED: Data transformation is 100% complete, correct, and secure!")
    print("="*60)

# ==============================================================================
# STEP 3: DATA TRANSFORMATION WORKFLOW
# ==============================================================================
file_2017 = 'wave-6-2016-shocksclean.dta' 
output_file = 'wave-6-shocks_2016_raw_preserved.csv'

print("📥 STEP 2: Loading Stata file with PyReadStat...")
df_17, meta = pyreadstat.read_dta(file_2017)

def full_clean_wave7_2017_exact_preserved(df):
    print("🧹 STEP 3: Executing Harmonization & Splitting Coping Dummies by Exact Codes...")
    
    # 1. จัดการคอลัมน์ QID บ่งชี้ตัวตนครัวเรือน
    if 'QID' in df.columns:
        pass
    elif 'qid' in df.columns:
        df = df.rename(columns={'qid': 'QID'})
    elif 'hhid' in df.columns:
        df['QID'] = df['hhid']
    else:
        df['QID'] = np.nan

    # 2. ปรับเปลี่ยนรหัสระบุ Missing ทั่วไป (-9, -99) ให้เป็น NaN 
    df = df.replace([-9, -99, -9.0, -99.0], np.nan)
    
    # 3. รักษาธรรมชาติคอลัมน์เงิน (ชื่อเดิมแยกส่วน) ล้างคำว่า 'none' ออกเป็น 0
    money_cols = ['_x31005a', '_x31005b', '_x31006a']
    for col in money_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace('none', '0', case=False).str.strip()
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # 4. รายการรหัสกิจกรรมจริงครบถ้วนสมบูรณ์ตามที่คุณส่งมา 100% (ไม่รวมรหัสข้ามตัวเลือก 60, 61, 98)
    exact_coping_codes = [
        # กลุ่มเลขหลักเดียว (1-9 และ 10)
        1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
        # กลุ่มเลข 10 และ 20 (ขายทรัพย์สิน และการกู้ยืมเงิน)
        11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27,
        # กลุ่มเลข 30 (เงินช่วยเหลือและการบริจาค)
        28, 29, 30, 31,
        # กลุ่มเลข 40 (การปรับตัวทางอาชีพ/ธุรกิจ แรงงานและการเกษตร)
        40, 41, 42, 43,
        # กลุ่มเลขบริการพื้นฐานและการป้องกันภัย (48-55)
        48, 49, 50, 51, 52, 53, 54, 55,
        # กลุ่มกิจกรรมร่วมและอื่นๆ (62, 63, 90)
        62, 63, 90
    ]
    
    coping_source_cols = ['_x31008', '_x31009', '_x31010']
    df_coping_numeric = df[coping_source_cols].apply(pd.to_numeric, errors='coerce')
    
    print("   -> Splitting exact coping codes into wide dummy columns (0 or 1)...")
    for code in exact_coping_codes:
        # บันทึกเป็น 1 ทันทีหากเจอตัวเลขอ้างอิงนี้ในช่องความสำคัญใดความสำคัญหนึ่งจากทั้งสามช่องดั้งเดิม
        df[f'coping_{code}'] = df_coping_numeric.isin([code]).any(axis=1).astype(int)
        
    # ลอจิกป้องกันความผิดพลาดเชิงสถิติ: หากครัวเรือนระบุช่องแรกเป็น 98 (No Answer) ให้ Dummy ทั้งหมดในแถวนั้นเป็น NaN
    invalid_mask = df_coping_numeric['_x31008'].isin([98, 98.0])
    for code in exact_coping_codes:
        df.loc[invalid_mask, f'coping_{code}'] = np.nan

    # 5. คงค่าธรรมชาติคอลัมน์ระยะเวลาการฟื้นตัวดิบดั้งเดิมไว้ (_x31012a และ _x31012)
    if '_x31012a' in df.columns:
        df['_x31012a'] = pd.to_numeric(df['_x31012a'], errors='coerce')
        df['_x31012a'] = df['_x31012a'].replace([98, 99, 98.0, 99.0], np.nan)

    # 6. จัดกลุ่มประเภทภัยพิบัติภาพกว้าง (Shock Grouping) สำหรับสถิติพื้นฐาน
    df['survey_year'] = 2016
    if '_x31002' in df.columns:
        shock_mapping = {
            10: 'agricultural', 11: 'agricultural', 55: 'agricultural', 63: 'agricultural',
            1: 'demographic', 2: 'demographic', 3: 'demographic', 24: 'demographic',
            5: 'economics', 6: 'economics', 18: 'economics', 21: 'economics', 22: 'economics', 62: 'economics',
            8: 'social', 70: 'social', 77: 'economics'
        }
        shock_col = pd.to_numeric(df['_x31002'], errors='coerce')
        df['shocks_Group'] = shock_col.map(shock_mapping).fillna('others')

    # 7. ย้ายตำแหน่งคอลุมน์ระบุตัวตนหลัก (QID, hhid) มาล็อกไว้ที่ตำแหน่ง 2 คอลัมน์แรกสุดของตาราง
    remaining_cols = [col for col in df.columns if col not in ['QID', 'hhid']]
    ordered_cols = ['QID', 'hhid'] + remaining_cols
    df = df[ordered_cols]
    print("   -> Successfully rearranged columns: ['QID', 'hhid'] are now at the front.")

    return df

# เริ่มการรันกระบวนการแปลงข้อมูลทั้งหมด
df_17 = full_clean_wave7_2017_exact_preserved(df_17)

# ==============================================================================
# STEP 4: EXPORT & RUN INTEGRITY CHECK
# ==============================================================================
print(f"💾 STEP 4: Exporting preserved dataset to CSV: '{output_file}'...")
df_17.to_csv(output_file, index=False, encoding='utf-8-sig')

# ตรวจทาน Data Integrity อัตโนมัติหลังเซฟเสร็จ
verify_data_integrity(df_original=df_17, csv_path=output_file)

In [ ]:
# ==============================================================================
# STEP 1: SETUP & LIBRARIES
# ==============================================================================
import pandas as pd
import numpy as np
import os
import warnings
from typing import Dict, Any

warnings.filterwarnings('ignore')

try:
    import pyreadstat
except ImportError:
    print("Installing 'pyreadstat' package for stable data loading...")
    os.system('pip install pyreadstat')
    import pyreadstat

# ==============================================================================
# STEP 2: DATA INTEGRITY VERIFICATION SYSTEM
# ==============================================================================
def profile_dataset(df: pd.DataFrame, name: str) -> Dict[str, Any]:
    return {
        "name": name,
        "shape": df.shape,
        "columns": list(df.columns),
        "null_counts": df.isnull().sum().to_dict()
    }

def verify_data_integrity(df_original: pd.DataFrame, csv_path: str):
    print("\n" + "="*60)
    print("🔍 STEP 5: STARTING DATA INTEGRITY VERIFICATION")
    print("="*60)
    
    df_csv = pd.read_csv(csv_path)
    orig_prof = profile_dataset(df_original, "Processed DataFrame")
    csv_prof = profile_dataset(df_csv, "Saved CSV File")
    has_critical_error = False
    
    # Check 1: ตรวจสอบขนาดข้อมูล
    print("\n[Check 1/4] Checking Data Dimensions...")
    if orig_prof["shape"] == csv_prof["shape"]:
        print(f"✅ Success: Dimensions match perfectly! Shape: {orig_prof['shape']}")
    else:
        print(f"❌ CRITICAL DISCREPANCY: Dimensions do not match!")
        has_critical_error = True

    # Check 2: ตรวจสอบตำแหน่ง 2 คอลัมน์แรก (QID, hhid)
    print("\n[Check 2/4] Checking Identifier Column Positions...")
    first_two_cols = list(df_csv.columns)[:2]
    if first_two_cols == ['QID', 'hhid']:
        print(f"✅ Success: 'QID' and 'hhid' are safely locked at the first two columns! {first_two_cols}")
    else:
        print(f"❌ MISMATCH: Columns are not in the requested order. Found: {first_two_cols}")
        has_critical_error = True

    # Check 3: ตรวจสอบจำนวนแถว
    print("\n[Check 3/4] Checking Row Counts Consistency...")
    if len(df_original) == len(df_csv):
        print("✅ Success: Total Row count matches perfectly across both sources.")
    else:
        print("❌ MISMATCH: Row count discrepancy detected!")
        has_critical_error = True

    # Check 4: ตรวจสอบค่าความถูกต้องตัวเลข
    print("\n[Check 4/4] Checking Numeric Values Consistency...")
    try:
        numeric_cols = df_original.select_dtypes(include=[np.number]).columns.intersection(list(df_csv.columns))
        all_numeric_match = True
        for col in numeric_cols:
            o_val = df_original[col].fillna(-99999).to_numpy()
            c_val = df_csv[col].fillna(-99999).to_numpy()
            if not np.isclose(o_val, c_val, rtol=1e-05, atol=1e-08).all():
                print(f"⚠️ WARNING: Small numeric variation detected in column: [{col}]")
                all_numeric_match = False
        if all_numeric_match:
            print("✅ Success: Data values are completely secure and consistent.")
    except Exception as e:
        print(f"❌ Could not complete values check: {e}")
        has_critical_error = True

    print("\n" + "="*60)
    print("📊 VERIFICATION SUMMARY")
    print("="*60)
    if has_critical_error:
        print("🚨 CRITICAL ALERT: Data discrepancies found! Check the log details above.")
    else:
        print("🎉 PASSED: Data transformation is 100% complete, correct, and secure!")
    print("="*60)

# ==============================================================================
# STEP 3: DATA TRANSFORMATION WORKFLOW
# ==============================================================================
file_2017 = 'wave-7-2017-shocksclean.dta' 
output_file = 'wave-7-shocks_2017_raw_preserved.csv'

print("📥 STEP 2: Loading Stata file with PyReadStat...")
df_17, meta = pyreadstat.read_dta(file_2017)

def full_clean_wave7_2017_exact_preserved(df):
    print("🧹 STEP 3: Executing Harmonization & Splitting Coping Dummies by Exact Codes...")
    
    # 1. จัดการคอลัมน์ QID บ่งชี้ตัวตนครัวเรือน
    if 'QID' in df.columns:
        pass
    elif 'qid' in df.columns:
        df = df.rename(columns={'qid': 'QID'})
    elif 'hhid' in df.columns:
        df['QID'] = df['hhid']
    else:
        df['QID'] = np.nan

    # 2. ปรับเปลี่ยนรหัสระบุ Missing ทั่วไป (-9, -99) ให้เป็น NaN 
    df = df.replace([-9, -99, -9.0, -99.0], np.nan)
    
    # 3. รักษาธรรมชาติคอลัมน์เงิน (ชื่อเดิมแยกส่วน) ล้างคำว่า 'none' ออกเป็น 0
    money_cols = ['_x31005a', '_x31005b', '_x31006a']
    for col in money_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace('none', '0', case=False).str.strip()
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # 4. รายการรหัสกิจกรรมจริงครบถ้วนสมบูรณ์ตามที่คุณส่งมา 100% (ไม่รวมรหัสข้ามตัวเลือก 60, 61, 98)
    exact_coping_codes = [
        # กลุ่มเลขหลักเดียว (1-9 และ 10)
        1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
        # กลุ่มเลข 10 และ 20 (ขายทรัพย์สิน และการกู้ยืมเงิน)
        11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27,
        # กลุ่มเลข 30 (เงินช่วยเหลือและการบริจาค)
        28, 29, 30, 31,
        # กลุ่มเลข 40 (การปรับตัวทางอาชีพ/ธุรกิจ แรงงานและการเกษตร)
        40, 41, 42, 43,
        # กลุ่มเลขบริการพื้นฐานและการป้องกันภัย (48-55)
        48, 49, 50, 51, 52, 53, 54, 55,
        # กลุ่มกิจกรรมร่วมและอื่นๆ (62, 63, 90)
        62, 63, 90
    ]
    
    coping_source_cols = ['_x31008', '_x31009', '_x31010']
    df_coping_numeric = df[coping_source_cols].apply(pd.to_numeric, errors='coerce')
    
    print("   -> Splitting exact coping codes into wide dummy columns (0 or 1)...")
    for code in exact_coping_codes:
        # บันทึกเป็น 1 ทันทีหากเจอตัวเลขอ้างอิงนี้ในช่องความสำคัญใดความสำคัญหนึ่งจากทั้งสามช่องดั้งเดิม
        df[f'coping_{code}'] = df_coping_numeric.isin([code]).any(axis=1).astype(int)
        
    # ลอจิกป้องกันความผิดพลาดเชิงสถิติ: หากครัวเรือนระบุช่องแรกเป็น 98 (No Answer) ให้ Dummy ทั้งหมดในแถวนั้นเป็น NaN
    invalid_mask = df_coping_numeric['_x31008'].isin([98, 98.0])
    for code in exact_coping_codes:
        df.loc[invalid_mask, f'coping_{code}'] = np.nan

    # 5. คงค่าธรรมชาติคอลัมน์ระยะเวลาการฟื้นตัวดิบดั้งเดิมไว้ (_x31012a และ _x31012)
    if '_x31012a' in df.columns:
        df['_x31012a'] = pd.to_numeric(df['_x31012a'], errors='coerce')
        df['_x31012a'] = df['_x31012a'].replace([98, 99, 98.0, 99.0], np.nan)

    # 6. จัดกลุ่มประเภทภัยพิบัติภาพกว้าง (Shock Grouping) สำหรับสถิติพื้นฐาน
    df['survey_year'] = 2017
    if '_x31002' in df.columns:
        shock_mapping = {
            10: 'agricultural', 11: 'agricultural', 55: 'agricultural', 63: 'agricultural',
            1: 'demographic', 2: 'demographic', 3: 'demographic', 24: 'demographic',
            5: 'economics', 6: 'economics', 18: 'economics', 21: 'economics', 22: 'economics', 62: 'economics',
            8: 'social', 70: 'social', 77: 'economics'
        }
        shock_col = pd.to_numeric(df['_x31002'], errors='coerce')
        df['shocks_Group'] = shock_col.map(shock_mapping).fillna('others')

    # 7. ย้ายตำแหน่งคอลุมน์ระบุตัวตนหลัก (QID, hhid) มาล็อกไว้ที่ตำแหน่ง 2 คอลัมน์แรกสุดของตาราง
    remaining_cols = [col for col in df.columns if col not in ['QID', 'hhid']]
    ordered_cols = ['QID', 'hhid'] + remaining_cols
    df = df[ordered_cols]
    print("   -> Successfully rearranged columns: ['QID', 'hhid'] are now at the front.")

    return df

# เริ่มการรันกระบวนการแปลงข้อมูลทั้งหมด
df_17 = full_clean_wave7_2017_exact_preserved(df_17)

# ==============================================================================
# STEP 4: EXPORT & RUN INTEGRITY CHECK
# ==============================================================================
print(f"💾 STEP 4: Exporting preserved dataset to CSV: '{output_file}'...")
df_17.to_csv(output_file, index=False, encoding='utf-8-sig')

# ตรวจทาน Data Integrity อัตโนมัติหลังเซฟเสร็จ
verify_data_integrity(df_original=df_17, csv_path=output_file)

In [ ]:
# ==============================================================================
# STEP 1: SETUP & LIBRARIES
# ==============================================================================
import pandas as pd
import numpy as np
import os
import warnings
from typing import Dict, Any

warnings.filterwarnings('ignore')

try:
    import pyreadstat
except ImportError:
    print("Installing 'pyreadstat' package for stable data loading...")
    os.system('pip install pyreadstat')
    import pyreadstat

# ==============================================================================
# STEP 2: DATA INTEGRITY VERIFICATION SYSTEM
# ==============================================================================
def profile_dataset(df: pd.DataFrame, name: str) -> Dict[str, Any]:
    return {
        "name": name,
        "shape": df.shape,
        "columns": list(df.columns),
        "null_counts": df.isnull().sum().to_dict()
    }

def verify_data_integrity(df_original: pd.DataFrame, csv_path: str):
    print("\n" + "="*60)
    print("🔍 STEP 5: STARTING DATA INTEGRITY VERIFICATION (WAVE 8 - REVISED)")
    print("="*60)
    
    df_csv = pd.read_csv(csv_path)
    orig_prof = profile_dataset(df_original, "Processed DataFrame")
    csv_prof = profile_dataset(df_csv, "Saved CSV File")
    has_critical_error = False
    
    # Check 1: ตรวจสอบขนาดข้อมูล
    print("\n[Check 1/4] Checking Data Dimensions...")
    if orig_prof["shape"] == csv_prof["shape"]:
        print(f"✅ Success: Dimensions match perfectly! Shape: {orig_prof['shape']}")
    else:
        print(f"❌ CRITICAL DISCREPANCY: Dimensions do not match!")
        has_critical_error = True

    # Check 2: ตรวจสอบตำแหน่ง 2 คอลัมน์แรก (QID, interview__key)
    print("\n[Check 2/4] Checking Identifier Column Positions...")
    first_two_cols = list(df_csv.columns)[:2]
    if first_two_cols == ['QID', 'interview__key']:
        print(f"✅ Success: 'QID' and 'interview__key' are safely locked at the front! {first_two_cols}")
    else:
        print(f"❌ MISMATCH: Column ordering incorrect. Found: {first_two_cols}")
        has_critical_error = True

    # Check 3: ตรวจสอบจำนวนแถว
    print("\n[Check 3/4] Checking Row Counts Consistency...")
    if len(df_original) == len(df_csv):
        print("✅ Success: Total Row count matches perfectly across both sources.")
    else:
        print("❌ MISMATCH: Row count discrepancy detected!")
        has_critical_error = True

    # Check 4: ตรวจสอบความถูกต้องและปริมาณค่าใน Dummy
    print("\n[Check 4/4] Checking Coping Dummy Distribution...")
    coping_cols = [c for c in df_csv.columns if c.startswith('coping_')]
    total_ones = df_csv[coping_cols].sum().sum()
    print(f"ℹ️ Total active coping strategies marked as '1' in output: {int(total_ones)} entries.")
    if total_ones == 0:
        print("❌ CRITICAL ERROR: Coping flags are all 0! Mapping logic failed.")
        has_critical_error = True
    else:
        print("✅ Success: Coping Dummy values are populated successfully.")

    print("\n" + "="*60)
    print("📊 VERIFICATION SUMMARY")
    print("="*60)
    if has_critical_error:
        print("🚨 CRITICAL ALERT: Data discrepancies found! Check details above.")
    else:
        print("🎉 PASSED: Wave 8 Coping alignment is now 100% accurate, complete, and verified!")
    print("="*60)

# ==============================================================================
# STEP 3: DATA TRANSFORMATION WORKFLOW WITH MATRIX COPING EXTRACTION
# ==============================================================================
file_2019 = 'wave-8-2019-shocksclean.dta' 
file_crosswalk = 'wave-8-2019-TVSEP2019.dta' # 💡 ใช้ไฟล์ที่คุณอัปโหลดประกบ QID
output_file = 'wave-8-shocks_2019_raw_preserved.csv'

print("📥 STEP 2: Loading Data files via PyReadStat...")
df_19, meta_19 = pyreadstat.read_dta(file_2019)
df_cw, meta_cw = pyreadstat.read_dta(file_crosswalk)

def full_clean_wave8_2019_matrix_mapping(df, df_cross):
    print("🧹 STEP 3: Executing Crosswalk Merging & Exact Coping Matrix Processing...")
    
    # 1. เชื่อมโยงดึงรหัส QID มาผูกผ่านตาราง Crosswalk
    df_cross['interview__key'] = df_cross['interview__key'].astype(str).str.strip()
    df['interview__key'] = df['interview__key'].astype(str).str.strip()
    
    crosswalk_sub = df_cross[['interview__key', 'QID']].drop_duplicates()
    df = pd.merge(df, crosswalk_sub, on='interview__key', how='left')
    print("   -> Successfully mapped QID onto the 2019 Shocks dataset.")

    # 2. ปรับเปลี่ยนรหัสระบุ Missing ทั่วไปในตารางดิบ
    df = df.replace([-9, -99, -9.0, -99.0], np.nan)
    
    # 3. รักษาธรรมชาติคอลัมน์เงิน ล้างคำว่า 'none' ออกเป็น 0
    money_cols = ['v31105a', 'v31105b', 'v31106a']
    for col in money_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace('none', '0', case=False).str.strip()
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # 4. 🔥 REVISED MATRIX MAPPING FOR COPING: กวาดล้างรหัสจริงจากคอลัมน์ v31108a__1 ถึง __7
    exact_coping_codes = [
        1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
        11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27,
        28, 29, 30, 31, 40, 41, 42, 43, 48, 49, 50, 51, 52, 53, 54, 55, 90
    ]
    
    coping_source_cols = [col for col in df.columns if col.startswith('v31108a__')]
    print(f"   -> Detected source columns for matching: {coping_source_cols}")
    
    if coping_source_cols:
        # แปลงเป็นตัวเลขล้วนเพื่อสแกนด้วย Numpy Array Matrix
        df_coping_raw = df[coping_source_cols].apply(lambda x: pd.to_numeric(x, errors='coerce'))
        coping_matrix = df_coping_raw.to_numpy()
        
        for code in exact_coping_codes:
            # วิ่งเช็กในแนวแถวข้อมูล: ถ้าเจอเลขรหัสเป๊ะๆ ในช่องใดช่องหนึ่ง คอลัมน์จะเป็น 1 ทันที
            df[f'coping_{code}'] = np.any(coping_matrix == code, axis=1).astype(int)
            
        # ลอจิกป้องกันสถิติเพี้ยน: หากช่องแรกเป็นค่าว่างหรือตอบ 98 (ไม่ตอบ) บังคับเซ็ต Dummy ทั้งหมดเป็น NaN
        if 'v31108a__1' in df.columns:
            invalid_mask = df_coping_raw['v31108a__1'].isin([98, 98.0]) | df_coping_raw['v31108a__1'].isna()
            for code in exact_coping_codes:
                df.loc[invalid_mask, f'coping_{code}'] = np.nan
    else:
        for code in exact_coping_codes:
            df[f'coping_{code}'] = 0

    # 5. คงค่าธรรมชาติคอลัมน์ระยะเวลาการฟื้นตัวของปี 2019 (v31112a)
    if 'v31112a' in df.columns:
        df['v31112a'] = pd.to_numeric(df['v31112a'], errors='coerce')
        df['v31112a'] = df['v31112a'].replace([98, 99, 98.0, 99.0], np.nan)

    # 6. เพิ่มตัวแปรปีประเมินผลและกลุ่มจัดประเภทภัยพิบัติภาพกว้าง (Shock Grouping)
    df['survey_year'] = 2019
    if 'v31102' in df.columns:
        shock_mapping = {
            10: 'agricultural', 11: 'agricultural', 55: 'agricultural', 63: 'agricultural',
            1: 'demographic', 2: 'demographic', 3: 'demographic', 24: 'demographic',
            5: 'economics', 6: 'economics', 18: 'economics', 21: 'economics', 22: 'economics', 62: 'economics',
            8: 'social', 70: 'social', 77: 'economics'
        }
        shock_col = pd.to_numeric(df['v31102'], errors='coerce')
        df['shocks_Group'] = shock_col.map(shock_mapping).fillna('others')

    # 7. ย้ายตำแหน่งคอลัมน์ระบุตัวตนหลัก (QID, interview__key) มาล็อกไว้ที่ตำแหน่งหน้าสุด 2 ช่องแรกของตาราง
    remaining_cols = [col for col in df.columns if col not in ['QID', 'interview__key']]
    ordered_cols = ['QID', 'interview__key'] + remaining_cols
    df = df[ordered_cols]
    print("   -> Successfully locked ['QID', 'interview__key'] at the front positions.")

    return df

# ดำเนินการประมวลผลเปลี่ยนชื่อคอลัมน์ตรงตามชุดข้อมูลแท้จริง
df_19 = full_clean_wave8_2019_matrix_mapping(df_19, df_cw)

# ==============================================================================
# STEP 4: EXPORT & RUN INTEGRITY CHECK
# ==============================================================================
print(f"💾 STEP 4: Exporting exact mapped dataset directly to CSV: '{output_file}'...")
df_19.to_csv(output_file, index=False, encoding='utf-8-sig')

# เรียกใช้งานระบบการตรวจสอบความถูกต้องเสร็จสิ้นกระบวนการ
verify_data_integrity(df_original=df_19, csv_path=output_file)

In [ ]:
# ==============================================================================
# 1. SETUP & DATA LOADING (โหลดครบทั้ง 3 ไฟล์)
# ==============================================================================
import pandas as pd
import numpy as np
import pyreadstat
import warnings
warnings.filterwarnings('ignore')

# ประกาศชื่อไฟล์ดิบทั้ง 3 ตัวตามระบบ TVSEP 2022
file_detail = 'wave-9-2022-shocks_detail.dta'   # ไฟล์ที่ 1: รายละเอียด/เงิน/Coping
file_parent = 'wave-9-2022-shocks.dta'          # ไฟล์ที่ 2: รหัสประเภทภัยพิบัติ
file_crosswalk = 'wave-9-2022-TVSEP2022.dta' # ไฟล์ที่ 3: ทะเบียนกลางดึง QID (เปลี่ยนเป็น TVSEP2022.dta ได้ถ้ามี)
output_file = 'wave-9-shocks_2022_raw_preserved.csv'

print("📥 Loading all 3 required files...")
df_detail, meta_d = pyreadstat.read_dta(file_detail)
df_parent, meta_p = pyreadstat.read_dta(file_parent)
df_cw, meta_cw = pyreadstat.read_dta(file_crosswalk)

# ทำความสะอาด Key สำหรับการผูกข้อมูลกันพลาด
for table in [df_detail, df_parent, df_cw]:
    table['interview__key'] = table['interview__key'].astype(str).str.strip()

# ==============================================================================
# 2. DATA MERGING WORKFLOW (กระบวนการประกบ 3 ประสาน)
# ==============================================================================
print("🔀 Step 1: Merging parent shock types onto details...")
# ดึงข้อมูลรหัสประเภทภัยพิบัติจากตารางแม่มาใส่ตารางย่อย (เช็กชื่อคอลัมน์ภัยพิบัติ v31102b / v31102d)
parent_sub = df_parent[['interview__key', 'shocks__id', 'v31102b']].rename(columns={'v31102b': 'v31102d'})
df_cleaned = pd.merge(df_detail, parent_sub, on=['interview__key', 'shocks__id'], how='left')

print("🔀 Step 2: Merging QID from crosswalk registry...")
# ดึงรหัส QID มาผูกเข้าตารางหลักเพื่อเชื่อม Panel
cw_sub = df_cw[['interview__key', 'QID']].drop_duplicates()
df_cleaned = pd.merge(df_cleaned, cw_sub, on='interview__key', how='left')

# ==============================================================================
# 3. HARMONIZATION & COPING MATRIX (ล้างเงิน และ แกะรหัส Coping)
# ==============================================================================
print("🧹 Step 3: Cleaning money columns & extracting coping dummies...")
df_cleaned = df_cleaned.replace([-9, -99, -9.0, -99.0], np.nan)

# ล้างค่าเงินดิบดั้งเดิม
for col in ['v31105a', 'v31105b', 'v31106a']:
    if col in df_cleaned.columns:
        df_cleaned[col] = df_cleaned[col].astype(str).str.replace('none', '0', case=False).str.strip()
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce').fillna(0)

# แกะรหัสสากล Coping (1-90) จากกล่องลำดับความสำคัญ v31108a__1 ถึง __3 ของปี 2022
exact_coping_codes = [
    1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
    11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27,
    28, 29, 30, 31, 40, 41, 42, 43, 48, 49, 50, 51, 52, 53, 54, 55, 90
]
coping_source_cols = [c for c in df_cleaned.columns if c.startswith('v31108a__')]

if coping_source_cols:
    df_coping_raw = df_cleaned[coping_source_cols].apply(lambda x: pd.to_numeric(x, errors='coerce'))
    coping_matrix = df_coping_raw.to_numpy()
    for code in exact_coping_codes:
        df_cleaned[f'coping_{code}'] = np.any(coping_matrix == code, axis=1).astype(int)
    
    # ดักจับเคสไม่ตอบ (98) ให้เป็น NaN
    if 'v31108a__1' in df_cleaned.columns:
        invalid_mask = df_coping_raw['v31108a__1'].isin([98, 98.0]) | df_coping_raw['v31108a__1'].isna()
        for code in exact_coping_codes:
            df_cleaned.loc[invalid_mask, f'coping_{code}'] = np.nan

# ==============================================================================
# 4. FINAL POSITIONING & EXPORT (จัดระเบียบหัวตารางและส่งออก)
# ==============================================================================
df_cleaned['survey_year'] = 2022

# จัดกลุ่ม Shock Grouping
shock_mapping = {
    10: 'agricultural', 11: 'agricultural', 55: 'agricultural', 63: 'agricultural',
    1: 'demographic', 2: 'demographic', 3: 'demographic', 24: 'demographic',
    5: 'economics', 6: 'economics', 18: 'economics', 21: 'economics', 22: 'economics', 62: 'economics',
    8: 'social', 70: 'social', 77: 'economics'
}
if 'v31102d' in df_cleaned.columns:
    df_cleaned['shocks_Group'] = pd.to_numeric(df_cleaned['v31102d'], errors='coerce').map(shock_mapping).fillna('others')

# ล็อก QID และ interview__key ไว้หน้าสุด 2 ช่องแรก
remaining_cols = [c for c in df_cleaned.columns if c not in ['QID', 'interview__key']]
df_final = df_cleaned[['QID', 'interview__key'] + remaining_cols]

# เซฟออกเป็น CSV ตัวจบของปี 2022
df_final.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"🎉 SUCCESS: Saved {len(df_final)} complete shock entries to '{output_file}'!")
print(f"ℹ️ Check: Missing QID Count = {df_final['QID'].isna().sum()}")

In [ ]:
# ==============================================================================
# STEP 1: SETUP & LIBRARIES
# ==============================================================================
import pandas as pd
import numpy as np
import os
import warnings
from typing import Dict, Any

warnings.filterwarnings('ignore')

try:
    import pyreadstat
except ImportError:
    print("Installing 'pyreadstat' package for stable data loading...")
    os.system('pip install pyreadstat')
    import pyreadstat

# ==============================================================================
# STEP 2: DATA INTEGRITY VERIFICATION SYSTEM
# ==============================================================================
def profile_dataset(df: pd.DataFrame, name: str) -> Dict[str, Any]:
    return {
        "name": name,
        "shape": df.shape,
        "columns": list(df.columns),
        "null_counts": df.isnull().sum().to_dict()
    }

def verify_data_integrity(df_original: pd.DataFrame, csv_path: str):
    print("\n" + "="*60)
    print("🔍 STEP 5: STARTING DATA INTEGRITY VERIFICATION (WAVE 10 - 2024)")
    print("="*60)
    
    df_csv = pd.read_csv(csv_path)
    orig_prof = profile_dataset(df_original, "Processed DataFrame")
    csv_prof = profile_dataset(df_csv, "Saved CSV File")
    has_critical_error = False
    
    # Check 1: ตรวจสอบขนาดข้อมูล
    print("\n[Check 1/4] Checking Data Dimensions...")
    if orig_prof["shape"] == csv_prof["shape"]:
        print(f"✅ Success: Dimensions match perfectly! Shape: {orig_prof['shape']}")
    else:
        print(f"❌ CRITICAL DISCREPANCY: Dimensions do not match!")
        has_critical_error = True

    # Check 2: ตรวจสอบตำแหน่ง 2 คอลัมน์แรก (QID, interview__key)
    print("\n[Check 2/4] Checking Identifier Column Positions...")
    first_two_cols = list(df_csv.columns)[:2]
    if first_two_cols == ['QID', 'interview__key']:
        print(f"✅ Success: 'QID' and 'interview__key' are safely locked at the front! {first_two_cols}")
    else:
        print(f"❌ MISMATCH: Column ordering incorrect. Found: {first_two_cols}")
        has_critical_error = True

    # Check 3: ตรวจสอบจำนวนแถว
    print("\n[Check 3/4] Checking Row Counts Consistency...")
    if len(df_original) == len(df_csv):
        print("✅ Success: Total Row count matches perfectly across both sources.")
    else:
        print("❌ MISMATCH: Row count discrepancy detected!")
        has_critical_error = True

    # Check 4: ตรวจสอบความถูกต้องและปริมาณค่าใน Dummy Coping
    print("\n[Check 4/4] Checking Coping Dummy Distribution...")
    coping_cols = [c for c in df_csv.columns if c.startswith('coping_')]
    total_ones = df_csv[coping_cols].sum().sum()
    print(f"ℹ️ Total active coping strategies marked as '1' in output: {int(total_ones)} entries.")
    if total_ones == 0:
        print("❌ CRITICAL ERROR: Coping flags are all 0! Matrix mapping logic failed.")
        has_critical_error = True
    else:
        print("✅ Success: Coping Dummy values are populated successfully.")

    print("\n" + "="*60)
    print("📊 VERIFICATION SUMMARY")
    print("="*60)
    if has_critical_error:
        print("🚨 CRITICAL ALERT: Data discrepancies found! Check details above.")
    else:
        print("🎉 PASSED: Wave 10 (2024) Matrix Mapping to 'coping' is 100% aligned, accurate, and secure!")
    print("="*60)

# ==============================================================================
# STEP 3: DATA TRANSFORMATION WORKFLOW WITH 3-WAY FILE MERGING (2024)
# ==============================================================================
file_2024_detail = 'wave-10-2024-shocks_detail.dta'  # ไฟล์ที่ 1: รายละเอียด/เงิน/Coping
file_2024_parent = 'wave-10-2024-shocks.dta'         # ไฟล์ที่ 2: ดึงรหัสประเภทภัยพิบัติ (v31102b)
file_crosswalk = 'wave-10-2024-TVSEP2024.dta'       # ไฟล์ที่ 3: ดึงรหัส QID มาผูก (เปลี่ยนชื่อได้ถ้ามีของปี 2024)
output_file = 'wave-10-shocks_2024_raw_preserved.csv'

print("📥 STEP 2: Loading 2024 Data files via PyReadStat...")
df_detail, meta_d = pyreadstat.read_dta(file_2024_detail)
df_parent, meta_p = pyreadstat.read_dta(file_2024_parent)
df_cw, meta_cw = pyreadstat.read_dta(file_crosswalk)

def full_clean_wave10_2024_3way_mapping(df_d, df_p, df_cross):
    print("🧹 STEP 3: Executing 3-Way File Merging & Exact Coping Matrix Processing for 2024...")
    
    # ล้างช่องว่างของระบบ Key ป้องกันการ Merge พลาด
    for table in [df_d, df_p, df_cross]:
        if 'interview__key' in table.columns:
            table['interview__key'] = table['interview__key'].astype(str).str.strip()
            
    # ประสานที่ 1: ดึงคอลัมน์ภัยพิบัติ v31102b จากตารางแม่มาใส่ตารางรายละเอียดย่อย
    print("   -> Merging parent shock types onto details dataset...")
    # ในปี 2024 ตารางแม่มักจะไม่มีรหัส ID อื่นๆ ยกเว้น interview__key และ shocks__id 
    parent_sub = df_p[['interview__key', 'shocks__id', 'v31102b']].rename(columns={'v31102b': 'v31102d'})
    df = pd.merge(df_d, parent_sub, on=['interview__key', 'shocks__id'], how='left')
    
    # ประสานที่ 2: ดึงรหัส QID ดั้งเดิมจากทะเบียนกลาง Crosswalk มาประกบ
    print("   -> Merging QID from crosswalk registry...")
    crosswalk_sub = df_cross[['interview__key', 'QID']].drop_duplicates()
    df = pd.merge(df, crosswalk_sub, on='interview__key', how='left')
    print(f"   -> Combined Rows Count: {len(df)}")

    # 2. ปรับเปลี่ยนรหัสระบุ Missing ทั่วไปในตารางดิบ
    df = df.replace([-9, -99, -9.0, -99.0], np.nan)
    
    # 3. รักษาธรรมชาติคอลัมน์เงิน ล้างคำว่า 'none' ออกเป็น 0
    money_cols = ['v31105a', 'v31105b', 'v31106a']
    for col in money_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace('none', '0', case=False).str.strip()
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # 4. 🔥 MATRIX MAPPING FOR COPING: แกะรหัสสากล 1-90 จากคอลัมน์ v31108a__1 ถึง __3 ของปี 2024
    exact_coping_codes = [
        1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
        11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27,
        28, 29, 30, 31, 40, 41, 42, 43, 48, 49, 50, 51, 52, 53, 54, 55, 90
    ]
    
    coping_source_cols = [col for col in df.columns if col.startswith('v31108a__')]
    print(f"   -> Detected 2024 priority coping columns: {coping_source_cols}")
    
    if coping_source_cols:
        # แปลงเป็นตัวเลขล้วนเพื่อทำ Numpy Row Matrix Scan
        df_coping_raw = df[coping_source_cols].apply(lambda x: pd.to_numeric(x, errors='coerce'))
        coping_matrix = df_coping_raw.to_numpy()
        
        for code in exact_coping_codes:
            # ตรวจสอบแนวแถว: ถ้ามีรหัสนี้ปรากฏในกล่องลำดับความสำคัญช่องใดช่องหนึ่ง สลักคอลัมน์ใหม่เป็น 1
            df[f'coping_{code}'] = np.any(coping_matrix == code, axis=1).astype(int)
            
        # ลอจิกควบคุมความแม่นยำ: หากช่องแรกเป็นค่าว่างหรือตอบ 98 บังคับเซ็ต Dummy ทั้งเซ็ตของแถวนั้นเป็น NaN
        if 'v31108a__1' in df.columns:
            invalid_mask = df_coping_raw['v31108a__1'].isin([98, 98.0]) | df_coping_raw['v31108a__1'].isna()
            for code in exact_coping_codes:
                df.loc[invalid_mask, f'coping_{code}'] = np.nan
    else:
        for code in exact_coping_codes:
            df[f'coping_{code}'] = 0

    # 5. คงค่าธรรมชาติคอลัมน์ระยะเวลาการฟื้นตัวของปี 2024 (v31112a)
    if 'v31112a' in df.columns:
        df['v31112a'] = pd.to_numeric(df['v31112a'], errors='coerce')
        df['v31112a'] = df['v31112a'].replace([98, 99, 98.0, 99.0], np.nan)

    # 6. เพิ่มตัวแปรปีประเมินผลและกลุ่มจัดประเภทภัยพิบัติภาพกว้าง (Shock Grouping) ของปี 2024
    df['survey_year'] = 2024
    if 'v31102d' in df.columns:
        shock_mapping = {
            10: 'agricultural', 11: 'agricultural', 55: 'agricultural', 63: 'agricultural',
            1: 'demographic', 2: 'demographic', 3: 'demographic', 24: 'demographic',
            5: 'economics', 6: 'economics', 18: 'economics', 21: 'economics', 22: 'economics', 62: 'economics',
            8: 'social', 70: 'social', 77: 'economics'
        }
        shock_col = pd.to_numeric(df['v31102d'], errors='coerce')
        df['shocks_Group'] = shock_col.map(shock_mapping).fillna('others')

    # 7. ย้ายตำแหน่งคอลัมน์ระบุตัวตนหลัก (QID, interview__key) มาล็อกไว้ที่ตำแหน่งหน้าสุด 2 ช่องแรกของตาราง
    remaining_cols = [col for col in df.columns if col not in ['QID', 'interview__key']]
    ordered_cols = ['QID', 'interview__key'] + remaining_cols
    df = df[ordered_cols]
    print("   -> Successfully locked ['QID', 'interview__key'] at the front positions.")

    return df

# เริ่มการรันลอจิก 3 ประสานผ่านฟังก์ชัน Matrix Mapping สำหรับปี 2024
df_24 = full_clean_wave10_2024_3way_mapping(df_detail, df_parent, df_cw)

# ==============================================================================
# STEP 4: EXPORT & RUN INTEGRITY CHECK
# ==============================================================================
print(f"💾 STEP 4: Exporting exact 2024 dataset directly to CSV: '{output_file}'...")
df_24.to_csv(output_file, index=False, encoding='utf-8-sig')

# เรียกใช้งานระบบการตรวจสอบความถูกต้องเสร็จสิ้นกระบวนการ
verify_data_integrity(df_original=df_24, csv_path=output_file)

In [ ]:
# 1. โหลดไฟล์ที่ Clean แล้วของทุกๆ ปีเข้ามาในระบบ
df_2007 = pd.read_csv('wave-1-shocks_2007_raw_preserved.csv')
df_2008 = pd.read_csv('wave-2-shocks_2008_raw_preserved.csv')
df_2010 = pd.read_csv('wave-3-shocks_2010_raw_preserved.csv')
df_2011 = pd.read_csv('wave-4-shocks_2011_raw_preserved.csv')
df_2013 = pd.read_csv('wave-5-shocks_2013_raw_preserved.csv')
df_2016 = pd.read_csv('wave-6-shocks_2016_raw_preserved.csv')
df_2017 = pd.read_csv('wave-7-shocks_2017_raw_preserved.csv')
df_2019 = pd.read_csv('wave-8-shocks_2019_raw_preserved.csv')
df_2022 = pd.read_csv('wave-9-shocks_2022_raw_preserved.csv')
df_2024 = pd.read_csv('wave-10-shocks_2024_raw_preserved.csv')

# 2. รวมไฟล์ทั้งหมดเข้าด้วยกันในแนวตั้ง (Vertical Append)
# สั่งให้คอลัมน์ชื่อเดียวกัน (เช่น QID, survey_year, coping_1 ถึง coping_90) วิ่งเข้าล็อกช่องเดียวกันโดยอัตโนมัติ
all_waves = [df_2007, df_2008, df_2010, df_2011, df_2013, df_2016, df_2017, df_2019, df_2022, df_2024]
df_master_panel = pd.concat(all_waves, ignore_index=True)

# 3. ล็อก QID และ survey_year ไว้ที่ 2 คอลัมน์แรกสุดเพื่อความสวยงาม
cols = list(df_master_panel.columns)
for c in ['QID', 'survey_year']:
    if c in cols: cols.remove(c)
df_master_panel = df_master_panel[['QID', 'survey_year'] + cols]

# 4. Export ออกเป็นไฟล์ Master Panel ตัวจบ
df_master_panel.to_csv('tvsep_shocks_master_panel_2007_2024.csv', index=False, encoding='utf-8-sig')

Running to panel data

In [ ]:
import pandas as pd
import numpy as np
import glob
import os

# 1. กำหนดโฟลเดอร์ที่เก็บไฟล์ CSV ที่ล้างเสร็จแล้วของทุกๆ ปี (2007-2024)
# 💡 แนะนำให้นำไฟล์ shocks_2017_raw_preserved.csv, shocks_2019_fixed_final.csv, 2022, 2024 ฯลฯ มารวมกันไว้ในโฟลเดอร์นี้
cleaned_folder = 'cleaned_shocks_waves'
master_output = 'TVSEP_Shocks_Unbalanced_Master_Panel_2007_2024.csv'

# ค้นหาไฟล์ .csv ทั้งหมดในโฟลเดอร์
file_list = sorted(glob.glob(os.path.join(cleaned_folder, "*.csv")))
print(f"📦 พบไฟล์ข้อมูลรายปีทั้งหมด {len(file_list)} ไฟล์ พร้อมสำหรับรวม Panel")

all_years_data = []

# 2. โหลดและปรับโครงสร้างเบื้องต้นรายไฟล์
for file_path in file_list:
    fname = os.path.basename(file_path)
    # บังคับคีย์หลักเป็น String เพื่อป้องกันปัญหาเลข 0 นำหน้าหาย
    df = pd.read_csv(file_path, dtype={'QID': str, 'interview__key': str})
    
    # เช็กเผื่อว่าปีไหนยังไม่มีคอลัมน์ survey_year ให้สกัดจากชื่อไฟล์
    if 'survey_year' not in df.columns:
        for year in range(2007, 2026):
            if str(year) in fname:
                df['survey_year'] = year
                break
                
    print(f"   -> โหลดสำเร็จ: {fname} | จำนวนข้อมูล: {len(df)} แถว")
    all_years_data.append(df)

# 3. รวมร่างแนวตั้ง (Vertical Append) เพื่อสร้าง Unbalanced Panel
print("\n🔀 กำลังผูกตารางแนวตั้งเข้าด้วยกัน...")
df_master = pd.concat(all_years_data, axis=0, ignore_index=True)

# 4. เติมค่า 0 ให้กับ Dummy Coping (coping_1 ถึง coping_90) ที่เป็น NaN
# เนื่องจากการทำ Unbalanced Panel ปีไหนไม่มีรหัสกิจกรรมนั้น ระบบจะมองเป็น NaN เราต้องแก้เป็น 0 (คือไม่ได้ทำ)
coping_cols = [c for c in df_master.columns if c.startswith('coping_')]
df_master[coping_cols] = df_master[coping_cols].fillna(0).astype(np.uint8)

# 5. จัดระเบียบคอลัมน์หลัก [QID, survey_year] ให้อยู่ 2 ช่องแรกสุดของตาราง
all_columns = list(df_master.columns)
for col_ident in ['QID', 'survey_year']:
    if col_ident in all_columns:
        all_columns.remove(col_ident)
df_master = df_master[['QID', 'survey_year'] + all_columns]

# 6. บันทึกผลลัพธ์เป็น Master CSV ตัวจบ
df_master.to_csv(master_output, index=False, encoding='utf-8-sig')

# ==============================================================================
# 📊 แสดงรายงานสถิติของ Unbalanced Panel หลังควบรวมเสร็จ
# ==============================================================================
print("\n" + "="*60)
print("🔍 REPORT: LONGITUDINAL UNBALANCED PANEL DATA SUMMARY")
print("="*60)
print(f"✅ จำนวนภัยพิบัติรวมในระบบ (Total Observations) : {len(df_master)} แถว")
print(f"✅ จำนวนครัวเรือนที่ไม่ซ้ำกันทั้งหมด (Unique HHs)   : {df_master['QID'].nunique()} ครัวเรือน")
print(f"✅ รายชื่อปีประเมินผลที่มีใน Panel ชุดนี้         : {sorted(df_master['survey_year'].unique().tolist())}")
print(f"✅ จำนวนคอลัมน์กิจกรรมการรับมือ (Coping Dummies)  : {len(coping_cols)} คอลัมน์ (coping_1 ถึง coping_90)")

# สรุปยอดภัยพิบัติแบ่งตามปีประเมินผล เพื่อเช็กว่าปีไหนขาดหายหรือไม่
print("\n📈 จำนวนภัยพิบัติที่บันทึกแยกรายปี (Observations per Year):")
print(df_master['survey_year'].value_counts().sort_index().to_string())
print("="*60)
print(f"🎉 เซฟไฟล์ Master Unbalanced Panel สำเร็จแล้วที่ -> '{master_output}'")

Data Harmonization and Coping Matrix Extraction for TVSEP Panel Data (2007-2024) has been completed successfully.

In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. โหลดไฟล์ Master Unbalanced Panel ที่มีข้อมูลกระจัดกระจายอยู่
master_csv = 'TVSEP_Shocks_Unbalanced_Master_Panel_2007_2024.csv'
refined_output = 'TVSEP_Shocks_Panel_Ready_To_Regress.csv'

print("📥 โหลดไฟล์ Master Unbalanced Panel...")
df = pd.read_csv(master_csv, dtype={'QID': str, 'interview__key': str})
print(f"   -> มิติข้อมูลดิบ: {df.shape[0]} แถว x {df.shape[1]} คอลัมน์")

# ==============================================================================
# 🔥 STEP 2: CREATING HARMONIZED CENTRAL COLUMNS (ยุบรวมคอลัมน์ขนาน)
# ==============================================================================
print("\n🧹 กำลังยุบรวมคอลัมน์ที่เก็บเรื่องเดียวกันข้ามระลอกปี...")

# A. รวมรหัสประเภทภัยพิบัติ (Shock Code)
# ปีเก่าใช้ _x31002, ปีใหม่ใช้ v31102d
df['shock_code'] = df['v31102d'].fillna(df['_x31002'])

# B. รวมมูลค่าความเสียหายหลัก (Loss Value)
# ดักจับคอลัมน์เงินปีเก่า และปีใหม่ มารวมกัน
loss_old = df['_x31005a'].fillna(df['_x31005']) if '_x31005a' in df.columns else df['_x31005']
df['loss_primary'] = df['v31105a'].fillna(loss_old).fillna(0)

# C. รวมระยะเวลาการฟื้นตัว (Recovery Status / Months)
recovery_old = df['_x31012a'].fillna(df['_x31012']) if '_x31012a' in df.columns else df['_x31012']
df['recovery_time'] = df['v31112a'].fillna(recovery_old)

# ==============================================================================
# 🔥 STEP 3: RE-ALIGNING COPING DUMMIES (ล้างค่าว่างใน Dummy)
# ==============================================================================
print("🛠️ ตรวจเช็กและเติมค่า 0 ในคอลัมน์ coping_1 ถึง coping_90 ทั่วทั้งตาราง...")
coping_cols = [c for c in df.columns if c.startswith('coping_')]
# เคส Unbalanced Panel แถวไหนที่ข้อมูล Coping โบ๋ ให้มองเป็น 0 (ไม่ได้ทำกิจกรรมรับมือนั้นๆ)
df[coping_cols] = df[coping_cols].fillna(0).astype(int)

# ==============================================================================
# 🔥 STEP 4: FINAL REARRANGEMENT & EXPORT (จัดคอลัมน์กลางขึ้นหน้าสุด)
# ==============================================================================
# ดึงเอาตัวแปรที่เราสร้างใหม่ไปล็อกไว้หน้าตาราง ร่วมกับ QID และ survey_year
central_identity = ['QID', 'survey_year', 'shock_code', 'loss_primary', 'recovery_time', 'shocks_Group']
remaining_cols = [c for c in df.columns if c not in central_identity]

df_panel_ready = df[central_identity + remaining_cols]

# 1. ยุบรวมรหัสภัยพิบัติให้อยู่ในคอลัมน์เดียวกันชื่อ 'shock_code'
df_panel_ready['shock_code'] = df['v31102d'].fillna(df['_x31002'])

# 2. ยุบรวมมูลค่าความเสียหายให้อยู่ในคอลัมน์เดียวกันชื่อ 'loss_value'
df_panel_ready['loss_value'] = df['v31105a'].fillna(df['_x31005a'].fillna(df['_x31005']))

# 3. ยุบรวมระยะเวลาฟื้นตัวให้อยู่ในคอลัมน์เดียวกันชื่อ 'recovery_time'
df_panel_ready['recovery_time'] = df['v31112a'].fillna(df['_x31012a'].fillna(df['_x31012']))

# ส่งออกเป็น CSV ตัวสุดท้ายที่จะเอาไปใส่ Stata หรือโมเดล Python
df_panel_ready.to_csv(refined_output, index=False, encoding='utf-8-sig')

print("\n" + "="*60)
print("📊 สรุปความพร้อมหลังจัดระเบียบ PANEL ตัวจบ")
print("="*60)
print(f"✅ มิติไฟล์สุดท้ายสำหรับการวิ่งโมเดล: {df_panel_ready.shape[0]} แถว x {df_panel_ready.shape[1]} คอลัมน์")
print(f"✅ ตัวแปรหลักที่พร้อมรัน Regression : {central_identity[:5]}")
print(f"✅ ยอดแถวจำแนกตามปีประเมินผล (Observations per Year):")
print(df_panel_ready['survey_year'].value_counts().sort_index().to_string())
print("="*60)
print(f"🎉 เซฟไฟล์พร้อมรันสมการ เรียบร้อยแล้วที่ -> '{refined_output}'")

📥 โหลดไฟล์ Master Unbalanced Panel...
   -> มิติข้อมูลดิบ: 23057 แถว x 233 คอลัมน์

🧹 กำลังยุบรวมคอลัมน์ที่เก็บเรื่องเดียวกันข้ามระลอกปี...
🛠️ ตรวจเช็กและเติมค่า 0 ในคอลัมน์ coping_1 ถึง coping_90 ทั่วทั้งตาราง...

📊 สรุปความพร้อมหลังจัดระเบียบ PANEL ตัวจบ
✅ มิติไฟล์สุดท้ายสำหรับการวิ่งโมเดล: 23057 แถว x 237 คอลัมน์
✅ ตัวแปรหลักที่พร้อมรัน Regression : ['QID', 'survey_year', 'shock_code', 'loss_primary', 'recovery_time']
✅ ยอดแถวจำแนกตามปีประเมินผล (Observations per Year):
survey_year
2007    2020
2008    3210
2010    2553
2011    2349
2013    2626
2016    3323
2017    1665
2019    2091
2022    2177
2024    1043
🎉 เซฟไฟล์พร้อมรันสมการ เรียบร้อยแล้วที่ -> 'TVSEP_Shocks_Panel_Ready_To_Regress.csv'
